# Do Index ETFs Help Small Investors Diversify?
## Evidence from India with Realistic Costs, Taxes and Whole-Unit Purchases

*Kalaiarasan G (Amrita School of Artificial Intelligence) and A. V. Shyam (Amrita School of Business), Amrita Vishwa Vidyapeetham, Coimbatore, India*

Replication notebook for the journal paper. It downloads **free data only**, builds survivorship-bias-free
random stock portfolios, simulates realistic small investors (lump sum and **SIP**, whole-unit purchases,
**Indian transaction costs and capital-gains tax**), and saves every table, figure, configuration, data
manifest and reference needed for the manuscript.

**How to run in Google Colab**
1. `Runtime → Run all`. Allow Google Drive access when asked; everything is saved to
   `MyDrive/ETF_SmallInvestor_India/`.
2. First run downloads data (≈ 20–40 min: ~140 NSE bhavcopy files plus Yahoo prices). Later runs reuse the cache.
3. Set `QUICK_RUN = True` in Section 1 for a fast first pass (fewer random draws and bootstrap replications).
   Use `QUICK_RUN = False` for the numbers reported in the paper.
4. `DATA_MODE = "synthetic"` runs the full pipeline on simulated data in a few minutes. It is a
   smoke test only and must never be used for results.

**Project folder (created automatically)**
```
ETF_SmallInvestor_India/
├── README.md                 how the project is organised and how to reproduce it
├── config/                   config.json (latest) + config_<run>.json (every run)
├── src/                      etf_engine.py, data_io.py, study.py (written by this notebook)
├── data/raw/                 untouched downloads (cache): etf_prices, etf_nav, index_tri,
│                             risk_free, bhavcopy, stock_prices
├── data/processed/           cleaned panels used in the analysis
├── data/download_manifest.jsonl   URL, file, time, SHA-256 and rows of every download
├── references/               references.bib, data_sources.md, tax_cost_rules.md,
│                             methodology_assumptions.md
├── results/run_<timestamp>/  tables/ (CSV, LaTeX, all_tables.xlsx), figures/ (PNG 300 dpi, PDF),
│                             logs/ (run log, package versions), key_results.md, output_index.csv
└── manuscript/               notes for writing (results summary copied here)
```

**Sections**
1. Settings · 2. Environment and folders · 3. Source code · 4. Configuration · 5. References and documentation ·
6. Data acquisition · 7. Data preparation · 8. Descriptive statistics · 9. ETF tracking quality and liquidity ·
10. Dynamic correlations · 11. Strategy weights · 12. ETF portfolio simulations · 13. Random direct-equity portfolios ·
14. Statistical tests · 15. Cost and tax erosion · 16. Sub-periods · 17. Robustness · 18. Additional figures ·
19. Export and key results

## 1. Settings

In [ ]:
DATA_MODE = "live"        # "live" = free data download | "synthetic" = smoke test only (never for results)
QUICK_RUN = False         # True = fewer random draws / bootstrap reps for a fast first pass
USE_DRIVE = True          # save everything to Google Drive (recommended)
PROJECT_NAME = "ETF_SmallInvestor_India"
AUTHOR = "Kalaiarasan G; A. V. Shyam"   # written into the configuration and README
FORCE_DOWNLOAD = False    # True = ignore the cache and re-download all data

## 2. Environment and project folders

In [ ]:
import os, sys, json, time, platform, subprocess, datetime as dt, warnings
warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "yfinance", "arch", "openpyxl"], check=False)

if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = f"/content/drive/MyDrive/{PROJECT_NAME}"
elif IN_COLAB:
    BASE = f"/content/{PROJECT_NAME}"
else:
    BASE = os.environ.get("ETF_BASE", os.path.abspath(PROJECT_NAME))

RUN_ID = dt.datetime.now().strftime("%Y%m%d_%H%M") + ("_synthetic" if DATA_MODE == "synthetic" else "") + ("_quick" if QUICK_RUN else "")
RUN_DIR = os.path.join(BASE, "results", f"run_{RUN_ID}")
DIRS = {
    "base": BASE, "src": f"{BASE}/src", "config": f"{BASE}/config", "references": f"{BASE}/references",
    "manuscript": f"{BASE}/manuscript", "raw": f"{BASE}/data/raw", "processed": f"{BASE}/data/processed",
    "raw_etf": f"{BASE}/data/raw/etf_prices", "raw_nav": f"{BASE}/data/raw/etf_nav",
    "raw_tri": f"{BASE}/data/raw/index_tri", "raw_rf": f"{BASE}/data/raw/risk_free",
    "raw_bhav": f"{BASE}/data/raw/bhavcopy", "raw_stock": f"{BASE}/data/raw/stock_prices",
    "results": RUN_DIR, "tables": f"{RUN_DIR}/tables", "figures": f"{RUN_DIR}/figures", "logs": f"{RUN_DIR}/logs",
}
if DATA_MODE == "synthetic":   # keep synthetic data away from real data
    DIRS["processed"] = f"{BASE}/data/processed_synthetic"
for p in DIRS.values():
    os.makedirs(p, exist_ok=True)
T0 = time.time()
print("Project folder:", BASE)
print("This run      :", RUN_DIR)

## 3. Source code
The three modules are written to `src/` so the exact code used for every run is archived with the project.

In [ ]:
open(os.path.join(DIRS['src'], 'etf_engine.py'), 'w').write(r'''"""
etf_engine.py
=============
Core engine for the study

    "Do Index ETFs Help Small Investors Diversify? Evidence from India with
     Realistic Costs, Taxes and Whole-Unit Purchases"
    Kalaiarasan G and A. V. Shyam, Amrita Vishwa Vidyapeetham, Coimbatore

Contents
--------
1. Indian transaction-cost model (brokerage, STT, exchange, SEBI, stamp, GST, DP, spread)
2. FIFO tax-lot engine for Indian capital-gains tax (current FY2026-27 rules and a
   historical, date-dependent regime used as a robustness check)
3. Portfolio simulator with whole-unit (integer) purchases, lump-sum and SIP
   contributions, rebalancing, tax payment each July, final liquidation
4. ETF weighting rules (1/N, minimum-variance, constrained max-Sharpe, risk parity)
   with sample, Ledoit-Wolf shrinkage and DCC-GARCH covariance
5. Performance metrics (Sharpe, Sortino, MDD, CVaR, CEQ, turnover, XIRR ...)
6. Statistical tests: Ledoit-Wolf (2008) Sharpe-difference bootstrap, Ledoit-Wolf
   (2011) variance-difference bootstrap, Romano-Wolf (2005) stepdown p-values
7. Tracking error / premium-discount / liquidity measures for ETFs

All monetary amounts are in Indian rupees (INR). Monthly frequency unless stated.
"""
from __future__ import annotations

import math
import warnings
from dataclasses import dataclass, field
from typing import Callable, Dict, List, Optional

import numpy as np
import pandas as pd
from scipy.optimize import brentq, minimize

EQUITY_TYPES = {"stock", "eq_etf"}
NONEQUITY_LISTED = {"gold_etf", "intl_etf"}
ASSET_TYPES = {"stock", "eq_etf", "gold_etf", "debt_etf", "intl_etf"}


# =============================================================================
# 1. Transaction costs
# =============================================================================
@dataclass
class CostModel:
    """Indian delivery-trade cost model. Rates are fractions of trade value.

    brokerage : "zero" (discount broker, free delivery) or "flat" (flat fee per order)
    """
    brokerage: str = "zero"
    flat_fee: float = 20.0
    stt: Dict[str, tuple] = field(default_factory=lambda: {
        "stock": (0.001, 0.001),      # 0.1% buy and sell (delivery)
        "eq_etf": (0.0, 0.00001),     # 0.001% on sale of equity-oriented fund units
        "gold_etf": (0.0, 0.0),
        "debt_etf": (0.0, 0.0),
        "intl_etf": (0.0, 0.0),
    })
    exchange: float = 0.0000297       # NSE cash-market transaction charge
    sebi: float = 0.000001            # Rs 10 per crore
    stamp_buy: float = 0.00015        # 0.015% on delivery purchases
    gst: float = 0.18                 # on brokerage + exchange + SEBI fee (+ DP)
    dp_charge_per_sell: float = 15.93 # depository charge per scrip per sell day (incl. GST)
    half_spread: Dict[str, float] = field(default_factory=lambda: {
        "stock": 0.0010, "eq_etf": 0.0005, "gold_etf": 0.0010,
        "debt_etf": 0.0015, "intl_etf": 0.0020})
    extra_bps: float = 0.0            # robustness: extra proportional cost (bps)
    frictionless: bool = False

    # fixed part of an order (does not scale with value)
    def fixed(self, side: str) -> float:
        if self.frictionless:
            return 0.0
        brk = self.flat_fee if self.brokerage == "flat" else 0.0
        dp = self.dp_charge_per_sell if side == "sell" else 0.0
        return brk * (1 + self.gst) + dp

    # proportional part of an order
    def prop(self, side: str, asset_type: str) -> float:
        if self.frictionless:
            return 0.0
        b, s = self.stt[asset_type]
        stt = b if side == "buy" else s
        stamp = self.stamp_buy if side == "buy" else 0.0
        return (stt + (self.exchange + self.sebi) * (1 + self.gst) + stamp
                + self.half_spread[asset_type] + self.extra_bps / 1e4)

    def breakdown(self, side: str, asset_type: str, value: float) -> Dict[str, float]:
        """Itemised cost of one order of `value` rupees."""
        if self.frictionless or value <= 0:
            return {"total": 0.0, "stt": 0.0}
        b, s = self.stt[asset_type]
        brk = self.flat_fee if self.brokerage == "flat" else 0.0
        out = {
            "brokerage": brk,
            "stt": value * (b if side == "buy" else s),
            "exchange": value * self.exchange,
            "sebi": value * self.sebi,
            "stamp": value * self.stamp_buy if side == "buy" else 0.0,
            "dp": self.dp_charge_per_sell if side == "sell" else 0.0,
            "spread": value * (self.half_spread[asset_type] + self.extra_bps / 1e4),
        }
        out["gst"] = self.gst * (out["brokerage"] + out["exchange"] + out["sebi"])
        out["total"] = sum(out.values())
        return out


# =============================================================================
# 2. Tax engine
# =============================================================================
DATE_LTCG_EQUITY = pd.Timestamp("2018-04-01")   # equity LTCG becomes taxable (10%)
DATE_GRANDFATHER = pd.Timestamp("2018-01-31")   # FMV date for grandfathering
DATE_FA2024 = pd.Timestamp("2024-07-23")        # Finance (No.2) Act 2024 rates
DATE_50AA = pd.Timestamp("2023-04-01")          # specified mutual fund rule
DATE_50AA_REDEF = pd.Timestamp("2025-04-01")    # 50AA narrowed to >65% debt funds

DEFAULT_CII = {2014: 240, 2015: 254, 2016: 264, 2017: 272, 2018: 280, 2019: 289,
               2020: 301, 2021: 317, 2022: 331, 2023: 348, 2024: 363, 2025: 376}


def fiscal_year(d: pd.Timestamp) -> int:
    """Indian financial year labelled by its starting calendar year (FY2025 = Apr25-Mar26)."""
    return d.year if d.month >= 4 else d.year - 1


@dataclass
class TaxEngine:
    """FIFO capital-gains tax for a resident individual.

    regime : "current"    -> FY2026-27 rules applied to every sale
             "historical" -> rules in force on each sale/purchase date
             "none"       -> no tax
    slab   : marginal income-tax slab rate for slab-taxed gains (debt, short-term gold/intl)
    """
    regime: str = "current"
    slab: float = 0.20
    cess: float = 0.04
    eq_st_rate: float = 0.20
    eq_lt_rate: float = 0.125
    eq_lt_exemption: float = 125000.0
    oth_lt_rate: float = 0.125
    cii: Dict[int, int] = field(default_factory=lambda: dict(DEFAULT_CII))
    gf_prices: Optional[Dict[str, float]] = None   # FMV on 31-Jan-2018 per asset
    realized: Dict[int, list] = field(default_factory=dict)
    cf_st: float = 0.0     # short-term loss carried forward
    cf_lt: float = 0.0     # long-term loss carried forward
    paid: Dict[int, float] = field(default_factory=dict)

    # --- classification of one realised lot ---------------------------------
    def _classify(self, atype, buy_date, sell_date, sale_value, cost_basis, asset, units=0.0):
        hold = (sell_date - buy_date).days
        if self.regime == "current":
            if atype in EQUITY_TYPES:
                return ("eq_lt", self.eq_lt_rate) if hold > 365 else ("eq_st", self.eq_st_rate), cost_basis
            if atype in NONEQUITY_LISTED:
                return ("oth_lt", self.oth_lt_rate) if hold > 365 else ("slab", self.slab), cost_basis
            return ("slab", self.slab), cost_basis           # debt ETF (Sec 50AA)

        # ---------------- historical regime ----------------
        if atype in EQUITY_TYPES:
            if hold > 365:
                if sell_date < DATE_LTCG_EQUITY:
                    return ("eq_lt_exempt", 0.0), cost_basis
                if buy_date <= DATE_GRANDFATHER and self.gf_prices and asset in self.gf_prices:
                    fmv = self.gf_prices[asset] * units
                    cost_basis = max(cost_basis, min(fmv, sale_value))
                rate = 0.10 if sell_date < DATE_FA2024 else 0.125
                return ("eq_lt", rate), cost_basis
            return ("eq_st", 0.15 if sell_date < DATE_FA2024 else 0.20), cost_basis

        # non-equity: gold, international, debt
        if atype == "debt_etf" and buy_date >= DATE_50AA:
            return ("slab", self.slab), cost_basis
        if atype in NONEQUITY_LISTED and buy_date >= DATE_50AA and sell_date < DATE_50AA_REDEF:
            return ("slab", self.slab), cost_basis
        if sell_date < DATE_FA2024:
            if hold > 1095:   # 36 months, 20% with indexation
                fb, fs = fiscal_year(buy_date), fiscal_year(sell_date)
                if fb in self.cii and fs in self.cii:
                    cost_basis = cost_basis * self.cii[fs] / self.cii[fb]
                return ("oth_lt_idx", 0.20), cost_basis
            return ("slab", self.slab), cost_basis
        if hold > 365:
            return ("oth_lt", 0.125), cost_basis
        return ("slab", self.slab), cost_basis

    def record(self, asset, atype, buy_date, sell_date, sale_value, cost_basis, units=0.0):
        if self.regime == "none":
            return
        (bucket, rate), basis = self._classify(atype, buy_date, sell_date, sale_value, cost_basis, asset, units)
        fy = fiscal_year(sell_date)
        self.realized.setdefault(fy, []).append((bucket, rate, sale_value - basis))

    def exemption(self, fy: int) -> float:
        if self.regime == "current":
            return self.eq_lt_exemption
        return 125000.0 if fy >= 2024 else 100000.0

    def settle(self, fy: int) -> float:
        """Compute tax payable for financial year `fy` (with set-off and carry-forward)."""
        if self.regime == "none" or fy in self.paid:
            return 0.0
        rows = self.realized.get(fy, [])
        net: Dict[tuple, float] = {}
        for b, r, g in rows:
            net[(b, r)] = net.get((b, r), 0.0) + g
        st_b = {"eq_st", "slab"}
        st_loss = self.cf_st + sum(-v for (b, r), v in net.items() if v < 0 and b in st_b)
        lt_loss = self.cf_lt + sum(-v for (b, r), v in net.items() if v < 0 and b not in st_b)
        pos = [[b, r, v] for (b, r), v in net.items() if v > 0 and b != "eq_lt_exempt"]
        pos.sort(key=lambda x: -x[1])
        for p in pos:                                  # LT losses -> LT gains only
            if p[0] not in st_b and lt_loss > 0:
                use = min(lt_loss, p[2]); p[2] -= use; lt_loss -= use
        for p in pos:                                  # ST losses -> any gains
            if st_loss > 0:
                use = min(st_loss, p[2]); p[2] -= use; st_loss -= use
        self.cf_st, self.cf_lt = st_loss, lt_loss
        ex = self.exemption(fy)
        for p in [q for q in pos if q[0] == "eq_lt"]:  # Sec 112A exemption
            use = min(ex, p[2]); p[2] -= use; ex -= use
        tax = sum(v * r for b, r, v in pos) * (1 + self.cess)
        self.paid[fy] = tax
        return tax


# =============================================================================
# 3. Portfolio simulator
# =============================================================================
@dataclass
class Lot:
    units: float
    cost_unit: float      # cost basis per unit (price + allowable costs)
    date: pd.Timestamp


class Portfolio:
    def __init__(self, assets: List[str], types: Dict[str, str], cost: CostModel,
                 tax: TaxEngine, integer: bool = True, min_order: float = 0.0):
        self.assets = list(assets)
        self.types = types
        self.cost = cost
        self.tax = tax
        self.integer = integer
        self.min_order = min_order
        self.units = {a: 0.0 for a in self.assets}
        self.lots: Dict[str, List[Lot]] = {a: [] for a in self.assets}
        self.cash = 0.0
        self.period_cost = 0.0
        self.period_traded = 0.0
        self.period_sold = 0.0
        self.total_cost = 0.0
        self.last_price: Dict[str, float] = {}

    # ---- valuation ---------------------------------------------------------
    def holdings_value(self, px: pd.Series) -> float:
        v = 0.0
        for a, u in self.units.items():
            if u:
                p = px.get(a, np.nan)
                v += u * (p if np.isfinite(p) else self.last_price.get(a, 0.0))
        return v

    def value(self, px: pd.Series) -> float:
        return self.holdings_value(px) + self.cash

    # ---- trading -----------------------------------------------------------
    def sell(self, a: str, units: float, price: float, date: pd.Timestamp):
        units = min(units, self.units[a])
        if units <= 1e-12:
            return
        atype = self.types[a]
        val = units * price
        bd = self.cost.breakdown("sell", atype, val)
        c = bd["total"]
        proceeds = val - c
        allow = c - bd["stt"]                    # STT is not deductible (Sec 48)
        rem = units
        while rem > 1e-12 and self.lots[a]:
            lot = self.lots[a][0]
            take = min(rem, lot.units)
            frac = take / units
            self.tax.record(a, atype, lot.date, date, val * frac - allow * frac, take * lot.cost_unit, take)
            lot.units -= take
            rem -= take
            if lot.units <= 1e-12:
                self.lots[a].pop(0)
        self.units[a] -= units
        if self.units[a] < 1e-9:
            self.units[a] = 0.0
        self.cash += proceeds
        self.period_cost += c
        self.total_cost += c
        self.period_traded += val
        self.period_sold += val

    def buy(self, a: str, units: float, price: float, date: pd.Timestamp):
        if units <= 1e-12:
            return
        atype = self.types[a]
        val = units * price
        bd = self.cost.breakdown("buy", atype, val)
        c = bd["total"]
        self.cash -= val + c
        self.units[a] += units
        self.lots[a].append(Lot(units, (val + c - bd["stt"]) / units, date))
        self.period_cost += c
        self.total_cost += c
        self.period_traded += val

    def liquidate_missing(self, px: pd.Series, date):
        """Sell holdings whose price is missing (delisted) at last known price."""
        for a, u in self.units.items():
            if u and not np.isfinite(px.get(a, np.nan)):
                self.sell(a, u, self.last_price.get(a, 0.0), date)

    def _buy_plan(self, target_val: Dict[str, float], px: pd.Series, cash: float,
                  single: bool = False) -> Dict[str, float]:
        """Whole-unit purchase plan that moves holdings toward target values using `cash`.
        single=True: one order only, the whole amount into the most underweight asset
        (how a small SIP investor avoids paying a flat fee on several orders)."""
        deficits = {}
        for a, tv in target_val.items():
            p = px.get(a, np.nan)
            if not np.isfinite(p) or p <= 0:
                continue
            d = tv - self.units[a] * p
            if d > 0:
                deficits[a] = d
        if not deficits:
            return {}
        if single:
            best, best_gap = None, -1.0
            for a, d in deficits.items():
                unit_cost = px[a] * (1 + self.cost.prop("buy", self.types[a]))
                if unit_cost + self.cost.fixed("buy") <= cash and d / max(target_val[a], 1e-9) > best_gap:
                    best, best_gap = a, d / max(target_val[a], 1e-9)
            if best is None:
                return {}
            unit_cost = px[best] * (1 + self.cost.prop("buy", self.types[best]))
            u = (cash - self.cost.fixed("buy")) / unit_cost
            u = math.floor(u) if self.integer else u
            return {best: u} if u > 0 else {}
        if not self.integer:
            tot = sum(deficits.values())
            plan = {}
            for a, d in deficits.items():
                pr = self.cost.prop("buy", self.types[a])
                spend = cash * d / tot if tot > cash else d
                plan[a] = max(0.0, spend / (px[a] * (1 + pr)))
            return plan

        def need(plan):
            s = 0.0
            for a, u in plan.items():
                if u > 0:
                    v = u * px[a]
                    s += v * (1 + self.cost.prop("buy", self.types[a])) + self.cost.fixed("buy")
            return s

        plan = {a: math.floor(d / px[a]) for a, d in deficits.items()}
        tot = need(plan)
        if tot > cash and tot > 0:
            f = cash / tot
            plan = {a: math.floor(u * f) for a, u in plan.items()}
        while need(plan) > cash and any(u > 0 for u in plan.values()):
            a = max(plan, key=lambda k: plan[k] * px[k])
            plan[a] -= 1
        # greedy top-up: one unit at a time to the largest remaining shortfall
        while True:
            spare = cash - need(plan)
            best, best_gap = None, 0.0
            for a, d in deficits.items():
                gap = d - plan.get(a, 0) * px[a]
                if gap < 0.5 * px[a]:
                    continue
                marg = px[a] * (1 + self.cost.prop("buy", self.types[a]))
                if plan.get(a, 0) == 0:
                    marg += self.cost.fixed("buy")
                if marg <= spare and gap / max(target_val[a], 1e-9) > best_gap:
                    best, best_gap = a, gap / max(target_val[a], 1e-9)
            if best is None:
                break
            plan[best] = plan.get(best, 0) + 1
        if self.min_order > 0:
            plan = {a: u for a, u in plan.items() if u * px[a] >= self.min_order}
        return {a: u for a, u in plan.items() if u > 0}

    def trade_to(self, w: pd.Series, px: pd.Series, date, full: bool = True, single: bool = False):
        """Move toward target weights `w`. full=True sells overweights; False uses cash only."""
        w = w[w.index.isin(self.assets)]
        valid = [a for a in w.index if np.isfinite(px.get(a, np.nan)) and px[a] > 0]
        w = w.loc[valid]
        if w.sum() <= 0:
            return
        w = w / w.sum()
        W = self.value(px)
        target_val = {a: float(w.get(a, 0.0)) * W for a in self.assets}
        if full:
            for a in self.assets:
                u = self.units[a]
                if not u:
                    continue
                p = px.get(a, np.nan)
                if not np.isfinite(p):
                    continue
                tgt_units = target_val.get(a, 0.0) / p
                if self.integer:
                    tgt_units = math.floor(tgt_units)
                excess = u - tgt_units
                # avoid tiny sells: only sell if excess value > 0.5% of wealth or target is zero
                if excess > 1e-9 and (excess * p > 0.005 * W or target_val.get(a, 0.0) == 0):
                    self.sell(a, excess, p, date)
        plan = self._buy_plan({a: target_val[a] for a in w.index}, px, max(self.cash, 0.0), single=single)
        for a, u in plan.items():
            self.buy(a, u, px[a], date)

    def raise_cash(self, amount: float, px: pd.Series, date):
        """Sell pro-rata to raise `amount` rupees (used to pay tax)."""
        hv = self.holdings_value(px)
        if hv <= 0:
            return
        frac = min(1.0, amount * 1.02 / hv)
        for a in self.assets:
            u = self.units[a]
            p = px.get(a, np.nan)
            if u and np.isfinite(p):
                s = u * frac
                s = math.ceil(s) if self.integer else s
                self.sell(a, min(s, u), p, date)


@dataclass
class SimResult:
    values: pd.Series          # end-of-month portfolio value (after trades and tax)
    contrib: pd.Series         # contribution invested at each date
    returns: pd.Series         # time-weighted monthly returns
    costs: pd.Series
    taxes: pd.Series
    traded: pd.Series
    sold: pd.Series            # value sold (rebalancing / tax / delisting sales)
    weights: pd.DataFrame      # end-of-month weights
    total_contrib: float
    terminal_pre_tax: float
    terminal_post_tax: float
    total_cost: float
    total_tax: float
    final_liq_tax: float
    xirr: float


def simulate(prices: pd.DataFrame, types: Dict[str, str], weight_fn: Callable,
             rebalance_dates: set, contributions: pd.Series, cost: CostModel,
             tax: TaxEngine, integer: bool = True, threshold: Optional[float] = None,
             tax_month: int = 7, liquidate_end: bool = True, sip_single_order: bool = False) -> SimResult:
    """Run one portfolio through time.

    prices         : month-end tradable prices (index = dates, columns = assets)
    weight_fn(t)   : returns target weights (pd.Series) at date index position t
    rebalance_dates: dates on which a full rebalance happens (the first date always trades)
    contributions  : rupees added at each date (lump sum -> only first date)
    threshold      : if given, also rebalance when any weight drifts > threshold
    sip_single_order: in contribution-only months place one order (most underweight asset)
    """
    dates = prices.index
    assets = list(prices.columns)
    pf = Portfolio(assets, types, cost, tax, integer=integer)
    vals, rets, costs_, taxes_, trd, sold, wts = [], [], [], [], [], [], []
    contrib = contributions.reindex(dates).fillna(0.0)
    target = None
    prev_v = None
    for t, d in enumerate(dates):
        px = prices.iloc[t]
        pf.period_cost = 0.0
        pf.period_traded = 0.0
        pf.period_sold = 0.0
        tax_paid = 0.0
        pf.liquidate_missing(px, d)
        for a in assets:
            if np.isfinite(px[a]):
                pf.last_price[a] = px[a]
        # pay last financial year's tax in July
        if d.month == tax_month:
            due = tax.settle(fiscal_year(d) - 1)
            if due > 0:
                if pf.cash < due:
                    pf.raise_cash(due - pf.cash, px, d)
                pf.cash -= due
                tax_paid += due
        c = float(contrib.iloc[t])
        pf.cash += c
        do_full = (t == 0) or (d in rebalance_dates)
        if do_full or target is None:
            target = weight_fn(t)
        if threshold is not None and not do_full and target is not None:
            W = pf.value(px)
            if W > 0:
                cur = pd.Series({a: pf.units[a] * px[a] / W if np.isfinite(px[a]) else 0 for a in assets})
                if (cur.reindex(target.index).fillna(0) - target).abs().max() > threshold:
                    do_full = True
        if do_full:
            pf.trade_to(target, px, d, full=True)
        elif c > 0:
            pf.trade_to(target, px, d, full=False, single=sip_single_order)
        v = pf.value(px)
        if prev_v is not None and prev_v > 0:
            rets.append((v - c) / prev_v - 1.0)
        else:
            rets.append(np.nan)
        prev_v = v
        vals.append(v)
        costs_.append(pf.period_cost)
        taxes_.append(tax_paid)
        trd.append(pf.period_traded)
        sold.append(pf.period_sold)
        wts.append({a: (pf.units[a] * px[a] / v if v > 0 and np.isfinite(px[a]) else 0.0) for a in assets})

    values = pd.Series(vals, index=dates)
    terminal_pre = values.iloc[-1]
    final_tax = 0.0
    if liquidate_end:
        d = dates[-1]
        px = prices.iloc[-1]
        for a in assets:
            if pf.units[a]:
                p = px[a] if np.isfinite(px[a]) else pf.last_price.get(a, 0.0)
                pf.sell(a, pf.units[a], p, d)
        for fy in sorted(set(tax.realized) | {fiscal_year(d)}):
            if fy not in tax.paid:
                final_tax += tax.settle(fy)
        terminal_post = pf.cash - final_tax
    else:
        terminal_post = terminal_pre
    total_tax = float(np.nansum(taxes_)) + final_tax
    cfs = -contrib.values.copy()
    cfs[-1] += terminal_post
    return SimResult(
        values=values, contrib=contrib, returns=pd.Series(rets, index=dates).iloc[1:],
        costs=pd.Series(costs_, index=dates), taxes=pd.Series(taxes_, index=dates),
        traded=pd.Series(trd, index=dates), sold=pd.Series(sold, index=dates), weights=pd.DataFrame(wts, index=dates),
        total_contrib=float(contrib.sum()), terminal_pre_tax=float(terminal_pre),
        terminal_post_tax=float(terminal_post), total_cost=float(pf.total_cost),
        total_tax=total_tax, final_liq_tax=final_tax, xirr=xirr_monthly(cfs))


def rebalance_schedule(dates: pd.DatetimeIndex, freq: str) -> set:
    """freq: 'M', 'Q', 'A' (anniversary of first date) or 'none'."""
    if freq == "none":
        return set()
    if freq == "M":
        return set(dates)
    step = {"Q": 3, "A": 12}[freq]
    return set(dates[::step])


def xirr_monthly(cfs: np.ndarray) -> float:
    """Annualised money-weighted return from equally spaced monthly cash flows."""
    t = np.arange(len(cfs))
    def npv(r):
        return np.sum(cfs / (1 + r) ** (t / 12.0))
    try:
        return brentq(npv, -0.99, 10.0)
    except Exception:
        return np.nan


# =============================================================================
# 4. Weighting rules and covariance estimators
# =============================================================================
def cov_sample(R: pd.DataFrame) -> np.ndarray:
    return np.cov(R.values, rowvar=False)


def cov_ledoit_wolf(R: pd.DataFrame) -> np.ndarray:
    from sklearn.covariance import LedoitWolf
    return LedoitWolf().fit(R.values).covariance_


def _garch11_fit(x: np.ndarray):
    from arch import arch_model
    am = arch_model(x, mean="Constant", vol="GARCH", p=1, q=1, dist="normal", rescale=False)
    res = am.fit(disp="off", show_warning=False)
    std_resid = res.std_resid
    fvar = res.forecast(horizon=1, reindex=False).variance.values[-1, 0]
    return np.asarray(std_resid), float(fvar)


def dcc_forecast(Rd: pd.DataFrame, horizon_days: int = 21) -> np.ndarray:
    """One-step-ahead DCC(1,1)-GARCH(1,1) covariance from daily returns, scaled to a month.

    Engle (2002) two-step estimator: univariate GARCH(1,1), then DCC parameters by
    quasi-maximum likelihood on standardised residuals.
    """
    X = Rd.dropna().values * 100.0
    n, k = X.shape
    Z = np.empty_like(X)
    fv = np.empty(k)
    for i in range(k):
        Z[:, i], fv[i] = _garch11_fit(X[:, i])
    Z = np.nan_to_num(Z)
    S = np.corrcoef(Z, rowvar=False)

    def qlik(params):
        a, b = params
        if a < 0 or b < 0 or a + b >= 0.999:
            return 1e10
        Q = S.copy()
        ll = 0.0
        for t in range(1, n):
            z = Z[t - 1][:, None]
            Q = (1 - a - b) * S + a * (z @ z.T) + b * Q
            d = np.sqrt(np.diag(Q))
            R = Q / np.outer(d, d)
            try:
                sign, logdet = np.linalg.slogdet(R)
                ll += logdet + Z[t] @ np.linalg.solve(R, Z[t])
            except np.linalg.LinAlgError:
                return 1e10
        return 0.5 * ll

    res = minimize(qlik, x0=[0.03, 0.94], method="Nelder-Mead",
                   options={"xatol": 1e-4, "fatol": 1e-4, "maxiter": 300})
    a, b = res.x if res.fun < 1e9 else (0.0, 0.0)
    Q = S.copy()
    for t in range(1, n + 1):
        z = Z[t - 1][:, None]
        Q = (1 - a - b) * S + a * (z @ z.T) + b * Q
    d = np.sqrt(np.diag(Q))
    R = Q / np.outer(d, d)
    sd = np.sqrt(fv) / 100.0
    H = R * np.outer(sd, sd)
    return H * horizon_days


def w_equal(assets) -> pd.Series:
    return pd.Series(1.0 / len(assets), index=assets)


def _solve(obj, n, cap):
    cons = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]
    bnds = [(0.0, cap)] * n
    x0 = np.full(n, 1.0 / n)
    res = minimize(obj, x0, method="SLSQP", bounds=bnds, constraints=cons,
                   options={"maxiter": 500, "ftol": 1e-12})
    w = np.clip(res.x if res.success else x0, 0, None)
    return w / w.sum()


def w_minvar(cov: np.ndarray, assets, cap: float = 1.0) -> pd.Series:
    n = len(assets)
    cap = max(cap, 1.0 / n)
    return pd.Series(_solve(lambda w: w @ cov @ w, n, cap), index=assets)


def w_maxsharpe(mu: np.ndarray, cov: np.ndarray, rf: float, assets, cap: float = 1.0) -> pd.Series:
    n = len(assets)
    cap = max(cap, 1.0 / n)
    ex = mu - rf
    if np.all(ex <= 0):
        return w_minvar(cov, assets, cap)
    return pd.Series(_solve(lambda w: -(w @ ex) / np.sqrt(w @ cov @ w + 1e-16), n, cap), index=assets)


def w_riskparity(cov: np.ndarray, assets) -> pd.Series:
    n = len(assets)
    def obj(w):
        pv = w @ cov @ w
        rc = w * (cov @ w)
        return np.sum((rc - pv / n) ** 2) * 1e4
    return pd.Series(_solve(obj, n, 1.0), index=assets)


def diversification_ratio(w: pd.Series, cov: np.ndarray) -> float:
    w = w.values
    sd = np.sqrt(np.diag(cov))
    return float((w @ sd) / np.sqrt(w @ cov @ w))


# =============================================================================
# 5. Performance metrics
# =============================================================================
def max_drawdown(r: pd.Series) -> float:
    w = (1 + r.fillna(0)).cumprod()
    return float((w / w.cummax() - 1).min())


def perf_metrics(r: pd.Series, rf: pd.Series, gamma: float = 1.0) -> Dict[str, float]:
    r = r.dropna()
    rf = rf.reindex(r.index).fillna(0.0)
    ex = r - rf
    n = len(r)
    if n < 3:
        return {}
    ann_ret = float((1 + r).prod() ** (12.0 / n) - 1)
    ann_vol = float(r.std(ddof=1) * np.sqrt(12))
    sd_ex = ex.std(ddof=1)
    sharpe = float(ex.mean() / sd_ex * np.sqrt(12)) if sd_ex > 0 else np.nan
    dd = np.sqrt(np.mean(np.minimum(ex, 0) ** 2))
    sortino = float(ex.mean() / dd * np.sqrt(12)) if dd > 0 else np.nan
    q = np.quantile(r, 0.05)
    cvar = float(-r[r <= q].mean())
    ceq = float((ex.mean() - gamma / 2 * ex.var(ddof=1)) * 12)
    return {"ann_return": ann_ret, "ann_vol": ann_vol, "sharpe": sharpe, "sortino": sortino,
            "max_drawdown": max_drawdown(r), "cvar95_monthly": cvar, "ceq": ceq,
            "skew": float(r.skew()), "kurtosis": float(r.kurt()), "n_months": n}


def sim_summary(res: SimResult, rf: pd.Series) -> Dict[str, float]:
    m = perf_metrics(res.returns, rf)
    avg_v = res.values.replace(0, np.nan).mean()
    m.update({
        "total_contrib": res.total_contrib,
        "terminal_pre_tax": res.terminal_pre_tax,
        "terminal_post_tax": res.terminal_post_tax,
        "wealth_multiple_post_tax": res.terminal_post_tax / res.total_contrib if res.total_contrib else np.nan,
        "xirr_post_tax": res.xirr,
        "total_cost": res.total_cost,
        "total_tax": res.total_tax,
        "cost_pct_contrib": res.total_cost / res.total_contrib if res.total_contrib else np.nan,
        "tax_pct_contrib": res.total_tax / res.total_contrib if res.total_contrib else np.nan,
        # tax as a share of the gross gain (post-tax wealth + all tax paid - contributions)
        "tax_pct_gain": (res.total_tax / (res.terminal_post_tax + res.total_tax - res.total_contrib)
                         if (res.terminal_post_tax + res.total_tax - res.total_contrib) > 0 else np.nan),
        "annual_turnover": float(res.sold.sum() / avg_v / (len(res.values) / 12)) if avg_v else np.nan,
        "avg_cash_weight": float(1 - res.weights.sum(axis=1).mean()),
    })
    return m


# =============================================================================
# 6. Statistical tests
# =============================================================================
def _qs_kernel(x):
    x = np.asarray(x, dtype=float)
    out = np.ones_like(x)
    nz = x != 0
    z = 6 * np.pi * x[nz] / 5
    out[nz] = 25 / (12 * np.pi ** 2 * x[nz] ** 2) * (np.sin(z) / z - np.cos(z))
    return out


def hac_qs(Y: np.ndarray) -> np.ndarray:
    """HAC covariance (Quadratic-Spectral kernel, Andrews 1991 AR(1) bandwidth)."""
    T, k = Y.shape
    Y = Y - Y.mean(axis=0)
    num = den = 0.0
    for i in range(k):
        y = Y[:, i]
        if np.var(y) == 0:
            continue
        rho = np.clip(np.dot(y[1:], y[:-1]) / np.dot(y[:-1], y[:-1]), -0.97, 0.97)
        s2 = np.var(y[1:] - rho * y[:-1])
        num += 4 * rho ** 2 * s2 ** 2 / (1 - rho) ** 8
        den += s2 ** 2 / (1 - rho) ** 4
    alpha2 = num / den if den > 0 else 0.0
    bw = 1.3221 * (alpha2 * T) ** 0.2 if alpha2 > 0 else 1.0
    Psi = Y.T @ Y / T
    for j in range(1, T):
        kw = _qs_kernel(j / bw)
        if abs(kw) < 1e-8 and j > 5 * bw:
            break
        G = Y[j:].T @ Y[:-j] / T
        Psi += kw * (G + G.T)
    return Psi * T / (T - 4)


def _sr_stat(r1, r2):
    m1, m2 = r1.mean(), r2.mean()
    g1, g2 = (r1 ** 2).mean(), (r2 ** 2).mean()
    return m1 / np.sqrt(g1 - m1 ** 2) - m2 / np.sqrt(g2 - m2 ** 2), (m1, m2, g1, g2)


def _sr_grad(m1, m2, g1, g2):
    return np.array([g1 / (g1 - m1 ** 2) ** 1.5, -g2 / (g2 - m2 ** 2) ** 1.5,
                     -0.5 * m1 / (g1 - m1 ** 2) ** 1.5, 0.5 * m2 / (g2 - m2 ** 2) ** 1.5])


def _var_stat(r1, r2):
    m1, m2 = r1.mean(), r2.mean()
    g1, g2 = (r1 ** 2).mean(), (r2 ** 2).mean()
    return np.log(g1 - m1 ** 2) - np.log(g2 - m2 ** 2), (m1, m2, g1, g2)


def _var_grad(m1, m2, g1, g2):
    return np.array([-2 * m1 / (g1 - m1 ** 2), 2 * m2 / (g2 - m2 ** 2),
                     1 / (g1 - m1 ** 2), -1 / (g2 - m2 ** 2)])


def _cbb_indices(T, b, M, rng):
    l = int(np.ceil(T / b))
    starts = rng.integers(0, T, size=(M, l))
    idx = (starts[:, :, None] + np.arange(b)[None, None, :]) % T
    return idx.reshape(M, l * b), l


def _boot_test(r1, r2, stat_fn, grad_fn, b, M, seed):
    r1 = np.asarray(r1, float); r2 = np.asarray(r2, float)
    T = len(r1)
    d_hat, (m1, m2, g1, g2) = stat_fn(r1, r2)
    Y = np.column_stack([r1 - m1, r2 - m2, r1 ** 2 - g1, r2 ** 2 - g2])
    grad = grad_fn(m1, m2, g1, g2)
    se = np.sqrt(grad @ hac_qs(Y) @ grad / T)
    rng = np.random.default_rng(seed)
    idx, l = _cbb_indices(T, b, M, rng)
    a1, a2 = r1[idx], r2[idx]
    Tb = idx.shape[1]
    mm1, mm2 = a1.mean(1), a2.mean(1)
    gg1, gg2 = (a1 ** 2).mean(1), (a2 ** 2).mean(1)
    if stat_fn is _sr_stat:
        d_star = mm1 / np.sqrt(gg1 - mm1 ** 2) - mm2 / np.sqrt(gg2 - mm2 ** 2)
        G = np.column_stack([gg1 / (gg1 - mm1 ** 2) ** 1.5, -gg2 / (gg2 - mm2 ** 2) ** 1.5,
                             -0.5 * mm1 / (gg1 - mm1 ** 2) ** 1.5, 0.5 * mm2 / (gg2 - mm2 ** 2) ** 1.5])
    else:
        d_star = np.log(gg1 - mm1 ** 2) - np.log(gg2 - mm2 ** 2)
        G = np.column_stack([-2 * mm1 / (gg1 - mm1 ** 2), 2 * mm2 / (gg2 - mm2 ** 2),
                             1 / (gg1 - mm1 ** 2), -1 / (gg2 - mm2 ** 2)])
    Ys = np.stack([a1 - mm1[:, None], a2 - mm2[:, None],
                   a1 ** 2 - gg1[:, None], a2 ** 2 - gg2[:, None]], axis=2)  # M x Tb x 4
    blocks = Ys.reshape(M, l, b, 4).sum(axis=2) / np.sqrt(b)                  # M x l x 4
    Psi = np.einsum("mli,mlj->mij", blocks, blocks) / l
    se_star = np.sqrt(np.einsum("mi,mij,mj->m", G, Psi, G) / Tb)
    t_hat = abs(d_hat) / se
    t_star = np.abs(d_star - d_hat) / se_star
    p = (np.sum(t_star >= t_hat) + 1) / (M + 1)
    return {"diff": float(d_hat), "se": float(se), "t": float(d_hat / se), "p_value": float(p),
            "_dstar": d_star, "_sestar": se_star}


def ledoit_wolf_sharpe_test(r1, r2, b: int = 5, M: int = 4999, seed: int = 42) -> Dict[str, float]:
    """H0: equal Sharpe ratios (Ledoit & Wolf 2008, studentised circular block bootstrap).
    Input: excess returns. `diff` is the per-period (monthly) Sharpe difference."""
    out = _boot_test(r1, r2, _sr_stat, _sr_grad, b, M, seed)
    out["diff_annualised"] = out["diff"] * np.sqrt(12)
    return {k: v for k, v in out.items() if not k.startswith("_")}


def ledoit_wolf_variance_test(r1, r2, b: int = 5, M: int = 4999, seed: int = 42) -> Dict[str, float]:
    """H0: equal variances (Ledoit & Wolf 2011). `diff` = log(var1) - log(var2)."""
    out = _boot_test(r1, r2, _var_stat, _var_grad, b, M, seed)
    return {k: v for k, v in out.items() if not k.startswith("_")}


def romano_wolf_sharpe(bench: np.ndarray, others: Dict[str, np.ndarray], b: int = 5,
                       M: int = 4999, seed: int = 42) -> pd.DataFrame:
    """Romano-Wolf (2005) stepdown adjusted p-values for H0_j: SR_j = SR_bench (two-sided)."""
    names = list(others)
    t_obs, t_boot = [], []
    for nme in names:
        o = _boot_test(others[nme], bench, _sr_stat, _sr_grad, b, M, seed)  # same seed -> same blocks
        t_obs.append(abs(o["t"]))
        t_boot.append(np.abs(o["_dstar"] - o["diff"]) / o["_sestar"])
    t_obs = np.array(t_obs)
    t_boot = np.array(t_boot)          # k x M
    order = np.argsort(-t_obs)
    adj = np.empty(len(names))
    prev = 0.0
    for step, j in enumerate(order):
        active = order[step:]
        maxstat = t_boot[active].max(axis=0)
        p = (np.sum(maxstat >= t_obs[j]) + 1) / (M + 1)
        prev = max(prev, p)
        adj[j] = prev
    return pd.DataFrame({"strategy": names, "abs_t": t_obs, "rw_adj_p": adj})


# =============================================================================
# 7. ETF quality measures
# =============================================================================
def tracking_measures(r_etf: pd.Series, r_idx: pd.Series, periods: int = 252,
                      error_cut: float = 0.20) -> Dict[str, float]:
    """Three tracking-error definitions used in the Indian ETF literature.
    TE1 mean absolute difference, TE2 std of differences, TE3 regression std error.
    Days where ETF and index returns differ by more than `error_cut` (20%) are treated as
    data errors (bad ticks, unadjusted splits) and excluded; their number is reported."""
    df = pd.concat([r_etf, r_idx], axis=1).dropna()
    bad = (df.iloc[:, 0] - df.iloc[:, 1]).abs() > error_cut
    n_bad = int(bad.sum())
    df = df[~bad]
    if len(df) < 30:
        return {}
    e, i = df.iloc[:, 0].values, df.iloc[:, 1].values
    d = e - i
    X = np.column_stack([np.ones_like(i), i])
    beta, *_ = np.linalg.lstsq(X, e, rcond=None)
    resid = e - X @ beta
    te3 = np.sqrt(resid @ resid / (len(e) - 2))
    yrs = len(df) / periods
    return {
        "TE1_mad_ann": float(np.mean(np.abs(d)) * np.sqrt(periods)),
        "TE2_sd_ann": float(np.std(d, ddof=1) * np.sqrt(periods)),
        "TE3_reg_ann": float(te3 * np.sqrt(periods)),
        "tracking_diff_ann": float(np.prod(1 + e) ** (1 / yrs) - np.prod(1 + i) ** (1 / yrs)),
        "alpha_ann": float(beta[0] * periods), "beta": float(beta[1]),
        "n_obs": int(len(df)), "days_excluded_data_errors": n_bad,
    }


def premium_discount(price: pd.Series, nav: pd.Series) -> Dict[str, float]:
    df = pd.concat([price, nav], axis=1).dropna()
    if len(df) < 30:
        return {}
    pdisc = df.iloc[:, 0] / df.iloc[:, 1] - 1
    bad = pdisc.abs() > 0.5            # |premium| above 50% = NAV/price data error, not a real premium
    n_bad = int(bad.sum())
    pdisc = pdisc[~bad]
    return {"pd_days_excluded_data_errors": n_bad,
            "pd_mean": float(pdisc.mean()), "pd_sd": float(pdisc.std()),
            "pd_mean_abs": float(pdisc.abs().mean()),
            "pct_days_abs_gt_1pct": float((pdisc.abs() > 0.01).mean()),
            "pd_max": float(pdisc.max()), "pd_min": float(pdisc.min())}


def liquidity_measures(close: pd.Series, volume: pd.Series) -> Dict[str, float]:
    df = pd.concat([close, volume], axis=1).dropna()
    df.columns = ["p", "v"]
    tv = df["p"] * df["v"]
    r = df["p"].pct_change().abs()
    amihud = (r / (tv / 1e7)).replace([np.inf, -np.inf], np.nan)
    return {"avg_daily_traded_value_cr": float(tv.mean() / 1e7),
            "median_daily_traded_value_cr": float(tv.median() / 1e7),
            "amihud_illiq": float(amihud.mean()),
            "pct_zero_volume_days": float((df["v"] == 0).mean())}


def dcc_correlations(Rd: pd.DataFrame) -> pd.DataFrame:
    """Full-sample DCC(1,1)-GARCH(1,1) conditional correlations, one column per asset pair."""
    X = Rd.dropna()
    cols = list(X.columns)
    Xv = X.values * 100.0
    n, k = Xv.shape
    Z = np.empty_like(Xv)
    for i in range(k):
        Z[:, i], _ = _garch11_fit(Xv[:, i])
    Z = np.nan_to_num(Z)
    S = np.corrcoef(Z, rowvar=False)

    def path(a, b):
        Q = S.copy()
        out = np.empty((n, k, k))
        out[0] = S
        for t in range(1, n):
            z = Z[t - 1][:, None]
            Q = (1 - a - b) * S + a * (z @ z.T) + b * Q
            d = np.sqrt(np.diag(Q))
            out[t] = Q / np.outer(d, d)
        return out

    def qlik(p):
        a, b = p
        if a < 0 or b < 0 or a + b >= 0.999:
            return 1e10
        Rs = path(a, b)
        ll = 0.0
        for t in range(1, n):
            sign, logdet = np.linalg.slogdet(Rs[t])
            ll += logdet + Z[t] @ np.linalg.solve(Rs[t], Z[t])
        return 0.5 * ll

    res = minimize(qlik, x0=[0.03, 0.94], method="Nelder-Mead", options={"maxiter": 200})
    a, b = res.x if res.fun < 1e9 else (0.0, 0.0)
    Rs = path(a, b)
    data = {f"{cols[i]} / {cols[j]}": Rs[:, i, j] for i in range(k) for j in range(i + 1, k)}
    df = pd.DataFrame(data, index=X.index)
    df.attrs["dcc_a"], df.attrs["dcc_b"] = float(a), float(b)
    return df
''')
print('wrote src/etf_engine.py')

In [ ]:
open(os.path.join(DIRS['src'], 'data_io.py'), 'w').write(r'''"""
data_io.py
==========
Free-data acquisition for the ETF small-investor study, with on-disk caching and a
download manifest (URL, file, timestamp, SHA-256, row count) for reproducibility.

Free sources used
-----------------
* Yahoo Finance via `yfinance`  : ETF and stock daily prices (Close, Adj Close, Volume)
* NSE archives (bhavcopy)       : month-end snapshots of all listed equities, including
                                  stocks later delisted -> survivorship-bias-free universe
* mfapi.in (mirror of AMFI NAVs): daily NAV of each ETF scheme and of a liquid fund
* FRED (St. Louis Fed)          : India 3-month Treasury bill rate (INTGSTINM193N)
* niftyindices.com              : Total Return Index (TRI) of NIFTY indices

Every function caches its raw output under data/raw/... and never re-downloads a file
that already exists unless `force=True`.
"""
from __future__ import annotations

import datetime as dt
import hashlib
import io
import json
import os
import time
import zipfile
from typing import Dict, Iterable, List, Optional

import numpy as np
import pandas as pd

try:
    import requests
except ImportError:  # pragma: no cover
    requests = None

UA = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                   "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"),
    "Accept": "*/*",
    "Accept-Language": "en-US,en;q=0.9",
}


# -----------------------------------------------------------------------------
# Manifest
# -----------------------------------------------------------------------------
def sha256(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


class Manifest:
    """Append-only JSON-lines log of every file downloaded or derived."""

    def __init__(self, path: str):
        self.path = path
        os.makedirs(os.path.dirname(path), exist_ok=True)

    def log(self, source: str, url: str, file: str, rows: Optional[int] = None, note: str = ""):
        rec = {"utc": dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "source": source,
               "url": url, "file": file, "rows": rows, "note": note,
               "sha256": sha256(file) if file and os.path.exists(file) else None}
        with open(self.path, "a") as f:
            f.write(json.dumps(rec) + "\n")

    def table(self) -> pd.DataFrame:
        if not os.path.exists(self.path):
            return pd.DataFrame()
        return pd.read_json(self.path, lines=True)


def http_get(url: str, session=None, retries: int = 3, timeout: int = 40, pause: float = 1.0,
             method: str = "GET", **kw) -> Optional[bytes]:
    """GET/POST with retries. Returns None on 404 (e.g. a market holiday)."""
    s = session or requests.Session()
    for k in range(retries):
        try:
            r = s.request(method, url, headers={**UA, **kw.pop("headers", {})}, timeout=timeout, **kw)
            if r.status_code == 404:
                return None
            if r.status_code == 200 and r.content:
                return r.content
        except Exception:
            pass
        time.sleep(pause * (k + 1))
    return None


# -----------------------------------------------------------------------------
# Yahoo Finance
# -----------------------------------------------------------------------------
def yf_history(tickers: List[str], start: str, end: str, batch: int = 50) -> pd.DataFrame:
    """Daily OHLCV with both Close and Adj Close, long format."""
    import yfinance as yf
    out = []
    for i in range(0, len(tickers), batch):
        tk = tickers[i:i + batch]
        df = yf.download(tk, start=start, end=end, auto_adjust=False, actions=False,
                         group_by="ticker", progress=False, threads=True)
        if df is None or df.empty:
            continue
        for t in tk:
            try:
                sub = df[t] if isinstance(df.columns, pd.MultiIndex) else df
            except KeyError:
                continue
            sub = sub.dropna(how="all")
            if sub.empty:
                continue
            sub = sub.rename(columns=str.lower).rename(columns={"adj close": "adj_close"})
            sub = sub.reset_index().rename(columns={"Date": "date", "date": "date"})
            sub["ticker"] = t
            out.append(sub[["date", "ticker", "open", "high", "low", "close", "adj_close", "volume"]])
        time.sleep(1.0)
    if not out:
        return pd.DataFrame(columns=["date", "ticker", "open", "high", "low", "close", "adj_close", "volume"])
    res = pd.concat(out, ignore_index=True)
    res["date"] = pd.to_datetime(res["date"]).dt.tz_localize(None)
    return res


# -----------------------------------------------------------------------------
# mfapi.in (AMFI NAV history)
# -----------------------------------------------------------------------------
def mfapi_search(query: str) -> List[Dict]:
    raw = http_get("https://api.mfapi.in/mf/search", params={"q": query})
    return json.loads(raw) if raw else []


def mfapi_nav(code: int) -> pd.DataFrame:
    raw = http_get(f"https://api.mfapi.in/mf/{code}")
    if not raw:
        return pd.DataFrame(columns=["date", "nav"])
    js = json.loads(raw)
    df = pd.DataFrame(js.get("data", []))
    if df.empty:
        return pd.DataFrame(columns=["date", "nav"])
    df["date"] = pd.to_datetime(df["date"], format="%d-%m-%Y")
    df["nav"] = pd.to_numeric(df["nav"], errors="coerce")
    df = df.sort_values("date").reset_index(drop=True)
    df.attrs["scheme_name"] = js.get("meta", {}).get("scheme_name", "")
    return df[["date", "nav"]]


def pick_scheme(query: str, must: Iterable[str] = (), avoid: Iterable[str] = ()) -> Optional[Dict]:
    """Pick the first mfapi scheme whose name contains all `must` words and none of `avoid`."""
    for item in mfapi_search(query):
        name = item.get("schemeName", "").lower()
        if all(m.lower() in name for m in must) and not any(a.lower() in name for a in avoid):
            return item
    return None


# -----------------------------------------------------------------------------
# FRED
# -----------------------------------------------------------------------------
def fred_series(series_id: str) -> pd.DataFrame:
    raw = http_get(f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}")
    if not raw:
        return pd.DataFrame(columns=["date", "value"])
    df = pd.read_csv(io.BytesIO(raw))
    df.columns = ["date", "value"]
    df["date"] = pd.to_datetime(df["date"])
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    return df.dropna()


# -----------------------------------------------------------------------------
# niftyindices.com Total Return Index
# -----------------------------------------------------------------------------
def nifty_tri(index_name: str, start: str, end: str) -> pd.DataFrame:
    """Daily TRI from niftyindices.com, fetched one calendar year at a time."""
    s = requests.Session()
    s.headers.update({**UA, "Content-Type": "application/json; charset=UTF-8",
                      "Origin": "https://www.niftyindices.com",
                      "Referer": "https://www.niftyindices.com/reports/historical-data"})
    try:
        s.get("https://www.niftyindices.com/reports/historical-data", timeout=30)
    except Exception:
        pass
    url = "https://www.niftyindices.com/Backpage.aspx/getTotalReturnIndexString"
    rows = []
    y0, y1 = pd.Timestamp(start).year, pd.Timestamp(end).year
    for y in range(y0, y1 + 1):
        a = max(pd.Timestamp(start), pd.Timestamp(f"{y}-01-01")).strftime("%d-%b-%Y")
        b = min(pd.Timestamp(end), pd.Timestamp(f"{y}-12-31")).strftime("%d-%b-%Y")
        cinfo = "{'name':'%s','startDate':'%s','endDate':'%s','indexName':'%s'}" % (index_name, a, b, index_name)
        try:
            r = s.post(url, data=json.dumps({"cinfo": cinfo}), timeout=40)
            js = json.loads(json.loads(r.text)["d"])
            for rec in js:
                rows.append({"date": pd.to_datetime(rec["Date"]),
                             "tri": float(str(rec["TotalReturnsIndex"]).replace(",", ""))})
        except Exception:
            pass
        time.sleep(1.0)
    return pd.DataFrame(rows).drop_duplicates("date").sort_values("date").reset_index(drop=True) \
        if rows else pd.DataFrame(columns=["date", "tri"])


# -----------------------------------------------------------------------------
# NSE bhavcopy (month-end snapshots)
# -----------------------------------------------------------------------------
NSE_HOSTS = ["https://nsearchives.nseindia.com", "https://archives.nseindia.com"]
UDIFF_START = pd.Timestamp("2024-07-08")


def nse_session():
    s = requests.Session()
    s.headers.update({**UA, "Referer": "https://www.nseindia.com/"})
    try:
        s.get("https://www.nseindia.com", timeout=20)
    except Exception:
        pass
    return s


def _bhav_urls(d: pd.Timestamp) -> List[str]:
    mon = d.strftime("%b").upper()
    old = f"/content/historical/EQUITIES/{d.year}/{mon}/cm{d.strftime('%d')}{mon}{d.year}bhav.csv.zip"
    new = f"/content/cm/BhavCopy_NSE_CM_0_0_0_{d.strftime('%Y%m%d')}_F_0000.csv.zip"
    paths = [new, old] if d >= UDIFF_START else [old, new]
    return [h + p for p in paths for h in NSE_HOSTS]


def parse_bhavcopy(content: bytes, d: pd.Timestamp) -> pd.DataFrame:
    """Standardise old (cmDDMONYYYYbhav) and new (UDiFF) formats."""
    with zipfile.ZipFile(io.BytesIO(content)) as z:
        name = z.namelist()[0]
        df = pd.read_csv(z.open(name))
    df.columns = [c.strip() for c in df.columns]
    if "SYMBOL" in df.columns:
        out = pd.DataFrame({"symbol": df["SYMBOL"], "series": df["SERIES"].astype(str).str.strip(),
                            "isin": df.get("ISIN"), "close": df["CLOSE"],
                            "volume": df["TOTTRDQTY"], "traded_value": df["TOTTRDVAL"]})
    else:
        out = pd.DataFrame({"symbol": df["TckrSymb"], "series": df["SctySrs"].astype(str).str.strip(),
                            "isin": df.get("ISIN"), "close": df["ClsPric"],
                            "volume": df["TtlTradgVol"], "traded_value": df["TtlTrfVal"]})
    out["date"] = d
    return out


def fetch_bhavcopy(d: pd.Timestamp, session) -> Optional[pd.DataFrame]:
    for url in _bhav_urls(d):
        raw = http_get(url, session=session, retries=2, pause=0.8)
        if raw and raw[:2] == b"PK":
            try:
                df = parse_bhavcopy(raw, d)
                df.attrs["url"] = url
                return df
            except Exception:
                continue
    return None


def month_end_bhavcopies(start: str, end: str, out_dir: str, manifest: Manifest,
                         force: bool = False, max_back: int = 10) -> pd.DataFrame:
    """Download the last trading day's bhavcopy of every month in [start, end]."""
    os.makedirs(out_dir, exist_ok=True)
    session = nse_session()
    frames = []
    months = pd.date_range(start, end, freq="ME")
    for m in months:
        fpath = os.path.join(out_dir, f"bhav_{m.strftime('%Y%m')}.csv.gz")
        if os.path.exists(fpath) and not force:
            frames.append(pd.read_csv(fpath, parse_dates=["date"]))
            continue
        got = None
        for back in range(max_back):
            d = m - pd.Timedelta(days=back)
            if d.weekday() >= 5:
                continue
            got = fetch_bhavcopy(d, session)
            if got is not None:
                break
            time.sleep(0.5)
        if got is None:
            print(f"  ! no bhavcopy found for {m:%Y-%m}")
            continue
        url = got.attrs.get("url", "")
        got.to_csv(fpath, index=False, compression="gzip")
        manifest.log("NSE bhavcopy", url, fpath, len(got))
        frames.append(got)
        print(f"  bhavcopy {got['date'].iloc[0]:%Y-%m-%d}: {len(got)} rows")
        time.sleep(1.0)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def nifty500_current_list() -> pd.DataFrame:
    """Fallback universe (survivorship-biased) if NSE archives are unreachable."""
    raw = http_get("https://www.niftyindices.com/IndexConstituent/ind_nifty500list.csv")
    return pd.read_csv(io.BytesIO(raw)) if raw else pd.DataFrame()


# -----------------------------------------------------------------------------
# Stock panel construction
# -----------------------------------------------------------------------------
SPLIT_FACTORS = np.array([1.5, 2, 2.5, 3, 4, 5, 6, 10, 11, 20])


def clean_bhav_returns(close: pd.Series, tol: float = 0.06):
    """Monthly returns from raw closes, correcting splits/bonuses detected as
    price ratios near 1/k for common k. Returns (returns, n_corrections)."""
    r = close / close.shift(1)
    out = r.copy()
    n = 0
    for i, x in r.items():
        if not np.isfinite(x) or x > 0.6:
            continue
        k = 1 / x
        j = np.argmin(np.abs(SPLIT_FACTORS - k))
        if abs(SPLIT_FACTORS[j] - k) / SPLIT_FACTORS[j] < tol:
            out[i] = x * SPLIT_FACTORS[j]
            n += 1
    return out - 1, n


def build_stock_panel(bhav: pd.DataFrame, yf_monthly_adj: pd.DataFrame, top_n: int = 500,
                      min_price: float = 10.0, lookback: int = 3, min_yf_coverage: float = 0.9):
    """Return (prices_wide, eligible_wide, meta).

    bhav           : long month-end bhavcopy rows (date, symbol, series, isin, close, traded_value)
    yf_monthly_adj : wide month-end Adj Close, columns = Yahoo tickers (SYMBOL.NS)
    """
    b = bhav[bhav["series"].isin(["EQ", "BE", "BZ"])].copy()
    b["date"] = pd.to_datetime(b["date"]).dt.to_period("M").dt.to_timestamp("M")
    b["id"] = b["isin"].fillna(b["symbol"])
    b = b.sort_values("date").drop_duplicates(["date", "id"], keep="last")
    latest_sym = b.groupby("id")["symbol"].last()
    tv = b.pivot(index="date", columns="id", values="traded_value")
    close = b.pivot(index="date", columns="id", values="close")
    eq_only = b[b["series"] == "EQ"].pivot(index="date", columns="id", values="close").notna()
    tv_avg = tv.rolling(lookback, min_periods=1).mean()
    rank = tv_avg.where(eq_only & (close >= min_price)).rank(axis=1, ascending=False)
    eligible = (rank <= top_n)

    ids = eligible.columns[eligible.any()]
    prices = {}
    meta = []
    for i in ids:
        sym = latest_sym[i]
        tk = f"{sym}.NS"
        bh = close[i].dropna()
        src, ncorr = "bhav", 0
        if tk in yf_monthly_adj.columns:
            y = yf_monthly_adj[tk].reindex(bh.index)
            cov = y.notna().mean() if len(bh) else 0
            if cov >= min_yf_coverage:
                prices[i] = yf_monthly_adj[tk].reindex(close.index).where(close[i].notna())
                src = "yahoo_adj"
        if src == "bhav":
            r, ncorr = clean_bhav_returns(bh)
            lvl = (1 + r.fillna(0)).cumprod() * bh.iloc[0]
            prices[i] = lvl.reindex(close.index)
        meta.append({"id": i, "symbol": sym, "price_source": src, "split_corrections": ncorr,
                     "first": bh.index.min(), "last": bh.index.max(),
                     "months_eligible": int(eligible[i].sum())})
    P = pd.DataFrame(prices).reindex(columns=ids)
    meta = pd.DataFrame(meta)
    global LAST_RANK
    LAST_RANK = rank[ids]
    return P, eligible[ids], meta


# -----------------------------------------------------------------------------
# Synthetic data (smoke test only; never used for results)
# -----------------------------------------------------------------------------
def synthetic_dataset(etf_keys: List[str], start="2015-01-01", end="2026-09-30", n_stocks=700, seed=7):
    """Generate data with the same structure as the live pipeline's processed outputs."""
    rng = np.random.default_rng(seed)
    days = pd.bdate_range(start, end)
    T = len(days)
    spec = {"eq_large": (0.12, 0.17), "eq_next50": (0.13, 0.21), "gold": (0.10, 0.14),
            "debt": (0.065, 0.04), "intl": (0.15, 0.22), "eq_mid": (0.15, 0.22)}
    k = len(etf_keys)
    C = np.full((k, k), 0.2)
    for a in range(k):
        for b_ in range(k):
            ka, kb = etf_keys[a], etf_keys[b_]
            if ka.startswith("eq") and kb.startswith("eq"):
                C[a, b_] = 0.85
            if "gold" in (ka, kb):
                C[a, b_] = -0.05
            if "debt" in (ka, kb):
                C[a, b_] = 0.05
        C[a, a] = 1
    L = np.linalg.cholesky(C + 1e-6 * np.eye(k))
    mu = np.array([spec.get(x, (0.1, 0.15))[0] for x in etf_keys]) / 252
    sd = np.array([spec.get(x, (0.1, 0.15))[1] for x in etf_keys]) / np.sqrt(252)
    Z = rng.standard_normal((T, k)) @ L.T
    R = mu + sd * Z
    tri = pd.DataFrame(100 * np.cumprod(1 + R, axis=0), index=days, columns=etf_keys)
    nav = tri * np.exp(-np.arange(T)[:, None] * 0.0005 / 252)       # expense drag
    nav = nav * (1 + rng.normal(0, 0.0003, nav.shape))                 # tracking noise
    price = nav * (1 + rng.normal(0, 0.002, nav.shape))                # premium/discount
    vol = pd.DataFrame(rng.lognormal(12, 1, (T, k)), index=days, columns=etf_keys)
    etf_daily = []
    for key in etf_keys:
        df = pd.DataFrame({"date": days, "key": key, "close": price[key].values,
                           "adj_close": price[key].values, "volume": vol[key].values.round()})
        etf_daily.append(df)
    etf_daily = pd.concat(etf_daily, ignore_index=True)
    nav_long = nav.stack().rename("nav").reset_index().rename(columns={"level_0": "date", "level_1": "key"})
    tri_long = tri.stack().rename("tri").reset_index().rename(columns={"level_0": "date", "level_1": "key"})
    months = pd.date_range(start, end, freq="ME")
    rf = pd.DataFrame({"date": months, "rf": 0.065 / 12 + rng.normal(0, 0.0003, len(months))})
    # stocks: one-factor model, random listing / delisting
    m = len(months)
    mkt = rng.normal(0.010, 0.05, m)
    beta = rng.uniform(0.6, 1.4, n_stocks)
    idio = rng.normal(0, 1, (m, n_stocks)) * rng.uniform(0.06, 0.14, n_stocks)
    ret = 0.001 + beta * mkt[:, None] + idio
    lvl = rng.uniform(50, 3000, n_stocks) * np.cumprod(1 + ret, axis=0)
    P = pd.DataFrame(lvl, index=months, columns=[f"SYN{i:04d}" for i in range(n_stocks)])
    first = rng.integers(0, m // 2, n_stocks) * (rng.random(n_stocks) < 0.25)
    last = np.where(rng.random(n_stocks) < 0.12, rng.integers(m // 3, m, n_stocks), m)
    for j, c in enumerate(P.columns):
        P.iloc[:first[j], j] = np.nan
        P.iloc[last[j]:, j] = np.nan
    tv = P.notna() * rng.lognormal(18, 1.5, n_stocks)
    rank = tv.where(P.notna()).rank(axis=1, ascending=False)
    elig = rank <= 500
    meta = pd.DataFrame({"id": P.columns, "symbol": P.columns, "price_source": "synthetic",
                         "split_corrections": 0})
    return {"etf_daily": etf_daily, "etf_nav": nav_long, "etf_tri": tri_long, "rf": rf,
            "stock_prices": P, "stock_eligible": elig, "stock_rank": rank, "stock_meta": meta}


TRI_DIAG: Dict[str, dict] = {}
LAST_RANK = None


def read_tri_files(raw_dir: str, index_name: str, verbose: bool = True, keep_file: bool = False) -> pd.DataFrame:
    """Read TRI CSVs saved manually from niftyindices.com (Historical Data -> Total Returns Index).
    Accepts one or many files named like NIFTY_50*.csv; any column containing 'date' and any
    column containing 'total' (or named 'tri') are used."""
    import glob
    stem = index_name.replace(" ", "_")
    frames, diag = [], []
    files = sorted(set(glob.glob(os.path.join(raw_dir, f"{stem}_*.csv")) + glob.glob(os.path.join(raw_dir, f"{stem}.csv"))))
    for f in files:   # "NIFTY_50_*.csv" deliberately does not match NIFTY_500 files
        if index_name == "NIFTY 50" and os.path.basename(f).upper().startswith("NIFTY_50_EQUAL"):
            continue
        df = pd.read_csv(f)
        cols = {c.lower().strip(): c for c in df.columns}
        dc = next((c for k, c in cols.items() if "date" in k), None)
        vc = next((c for k, c in cols.items() if "total" in k or k == "tri"), None)
        if dc is None or vc is None:
            print(f"  skipped {os.path.basename(f)}: needs a date column and a Total Returns column")
            continue
        one = pd.DataFrame({
            "date": pd.to_datetime(df[dc], dayfirst=True, errors="coerce"),
            "tri": pd.to_numeric(df[vc].astype(str).str.replace(",", ""), errors="coerce")}).dropna()
        one["file"] = os.path.basename(f)
        name_col = next((c for k, c in cols.items() if "index" in k and "name" in k), None)
        diag.append({"file": os.path.basename(f), "rows": len(one),
                     "first_date": one["date"].min(), "last_date": one["date"].max(),
                     "first_value": one.sort_values("date")["tri"].iloc[0] if len(one) else np.nan,
                     "last_value": one.sort_values("date")["tri"].iloc[-1] if len(one) else np.nan,
                     "index_name_in_file": str(df[name_col].iloc[0]) if name_col and len(df) else "",
                     "value_column": vc})
        frames.append(one)
    if not frames:
        return pd.DataFrame(columns=["date", "tri"])
    out = pd.concat(frames).sort_values("date")
    dups = out[out.duplicated("date", keep=False)]
    conflict = dups.groupby("date")["tri"].nunique()
    out = out.drop_duplicates("date").reset_index(drop=True)
    r = out["tri"].pct_change()
    jumps = out.loc[r.abs() > 0.15, ["date", "tri", "file"]].assign(daily_return=r[r.abs() > 0.15])
    if verbose:
        print(f"  {index_name}: files read")
        print(pd.DataFrame(diag).to_string(index=False))
        if (conflict > 1).any():
            print(f"  ! {int((conflict > 1).sum())} dates appear in two files with DIFFERENT values (overlapping downloads).")
        if len(jumps):
            print(f"  ! {len(jumps)} day(s) with a TRI move above 15% - almost always a wrong file "
                  f"(price index instead of TRI, another index, or a mis-read date):")
            print(jumps.to_string(index=False))
        else:
            print("  OK: no daily TRI move above 15% (largest genuine move: about -13% on 23-Mar-2020).")
    TRI_DIAG[index_name] = {"files": pd.DataFrame(diag), "jumps": jumps}   # not in .attrs: pandas concat breaks on DataFrame attrs
    return out[["date", "tri"]].assign(file=out["file"]) if keep_file else out[["date", "tri"]]


def normalize_nav_units(price: pd.Series, nav: pd.Series, tol: float = 0.20) -> pd.Series:
    """Put NAV on the same per-unit basis as the (split-adjusted) market price.

    Yahoo prices are back-adjusted for unit splits; AMFI NAVs are not. On each day the
    NAV/price ratio is snapped to the nearest split factor (1, 2, 5, 10, ... 1000); if it is
    within `tol` of a factor other than 1, NAV is divided by that factor."""
    df = pd.concat([price.rename("p"), nav.rename("n")], axis=1).dropna()
    if df.empty:
        return nav
    facs = np.array(sorted({a * 10 ** b for a in (1, 2, 5) for b in range(-3, 4)}))
    # 21-day centred median of the NAV/price ratio: robust to genuine premiums (e.g. Nasdaq ETF
    # trading 5-25% above NAV) and to isolated bad ticks; snapped to the nearest split factor.
    lk = np.log(df["n"] / df["p"]).rolling(21, center=True, min_periods=3).median().values
    idx = np.abs(lk[:, None] - np.log(facs)[None, :]).argmin(axis=1)
    c = facs[idx]
    ok = (np.abs(lk - np.log(c)) < np.log(1.4)) & (c != 1)
    adj = pd.Series(np.where(ok, c, 1.0), index=df.index)
    out = nav.copy()
    common = out.index.intersection(adj.index)
    out.loc[common] = out.loc[common] / adj.loc[common]
    # days with NAV but no price: carry the nearest factor
    rest = out.index.difference(adj.index)
    if len(rest):
        f = adj.reindex(out.index).ffill().bfill()
        out.loc[rest] = nav.loc[rest] / f.loc[rest]
    return out


def despike_daily(s: pd.Series, window: int = 11, thresh: float = 0.40):
    """Remove bad ticks from a daily price series: points more than `thresh` (40%) away from the
    centred rolling median of `window` days are set missing and interpolated. A diversified ETF
    cannot move 40% against its own two-week median and back. Returns (clean series, n days fixed)."""
    x = s.dropna()
    if len(x) < window:
        return s, 0
    m = x.rolling(window, center=True, min_periods=3).median()
    bad = (x / m - 1).abs() > thresh
    y = x.mask(bad).interpolate(method="time").ffill().bfill()
    out = s.copy()
    out.loc[y.index] = y
    return out, int(bad.sum())


def clean_monthly_spikes(P: pd.DataFrame, jump: float = 0.5, back: float = 0.2):
    """Fix one-month price spikes that reverse the next month (classic data error): if
    |r_t| > `jump` and the two-month return (1+r_t)(1+r_t+1)-1 is within +-`back`, the month-t
    price is replaced by the geometric mean of its neighbours. Returns (clean panel, count)."""
    r = P.pct_change(fill_method=None)
    two = (1 + r) * (1 + r.shift(-1)) - 1
    flag = (r.abs() > jump) & (two.abs() < back)
    fix = np.sqrt(P.shift(1) * P.shift(-1))
    return P.mask(flag, fix), int(flag.sum().sum())
''')
print('wrote src/data_io.py')

In [ ]:
open(os.path.join(DIRS['src'], 'study.py'), 'w').write(r'''"""
study.py
========
Study-level helpers: panel construction, strategy weight schedules, scenario runs,
random direct-equity portfolios, table/figure export.
"""
from __future__ import annotations

import json
import os
import time
from typing import Dict, List

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

import etf_engine as E

# ----------------------------------------------------------------------------
# Plot style: journal-ready, colour-blind-validated categorical order
# ----------------------------------------------------------------------------
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e4e3df"
STOCK_GREY = "#8a8984"


def set_style():
    matplotlib.rcParams.update({
        "figure.dpi": 110, "savefig.dpi": 300, "font.size": 9, "axes.titlesize": 10,
        "axes.labelsize": 9, "legend.fontsize": 8, "xtick.labelsize": 8, "ytick.labelsize": 8,
        "axes.edgecolor": INK2, "axes.labelcolor": INK, "text.color": INK,
        "xtick.color": INK2, "ytick.color": INK2, "axes.grid": True, "grid.color": GRID,
        "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
        "lines.linewidth": 1.6, "legend.frameon": False, "font.family": "DejaVu Sans",
        "pdf.fonttype": 42, "ps.fonttype": 42})


# ----------------------------------------------------------------------------
# Output registry
# ----------------------------------------------------------------------------
class Outputs:
    """Saves tables (CSV + LaTeX + Excel workbook) and figures (PNG + PDF) with an index."""

    def __init__(self, dirs: Dict[str, str]):
        self.dirs = dirs
        self.tables: Dict[str, pd.DataFrame] = {}
        self.index: List[Dict] = []

    def table(self, df: pd.DataFrame, name: str, caption: str, index: bool = True, digits: int = 4):
        df = df.copy()
        df.to_csv(os.path.join(self.dirs["tables"], f"{name}.csv"), index=index)
        try:
            tex = df.to_latex(index=index, float_format=lambda x: f"{x:.{digits}f}",
                              caption=caption, label=f"tab:{name}", escape=True, longtable=False)
            with open(os.path.join(self.dirs["tables"], f"{name}.tex"), "w") as f:
                f.write(tex)
        except Exception as e:  # pragma: no cover
            print("LaTeX export skipped:", e)
        self.tables[name] = df if index else df.set_index(df.columns[0])
        self.index.append({"type": "table", "name": name, "caption": caption})

    def figure(self, fig, name: str, caption: str):
        for ext in ("png", "pdf"):
            fig.savefig(os.path.join(self.dirs["figures"], f"{name}.{ext}"), bbox_inches="tight")
        self.index.append({"type": "figure", "name": name, "caption": caption})

    def write_workbook(self):
        path = os.path.join(self.dirs["tables"], "all_tables.xlsx")
        with pd.ExcelWriter(path) as xw:
            pd.DataFrame(self.index).to_excel(xw, sheet_name="index", index=False)
            for n, df in self.tables.items():
                df.to_excel(xw, sheet_name=n[:31])
        pd.DataFrame(self.index).to_csv(os.path.join(self.dirs["results"], "output_index.csv"), index=False)
        return path


# ----------------------------------------------------------------------------
# Panels
# ----------------------------------------------------------------------------
def wide_monthly(long: pd.DataFrame, key: str, value: str) -> pd.DataFrame:
    w = long.pivot_table(index="date", columns=key, values=value, aggfunc="last").sort_index()
    return w.resample("ME").last()


def wide_daily(long: pd.DataFrame, key: str, value: str) -> pd.DataFrame:
    return long.pivot_table(index="date", columns=key, values=value, aggfunc="last").sort_index()


def contributions(dates: pd.DatetimeIndex, mode: str, amount: float) -> pd.Series:
    s = pd.Series(0.0, index=dates)
    if mode == "lump":
        s.iloc[0] = amount
    else:
        s[:] = amount
    return s


# ----------------------------------------------------------------------------
# ETF weight schedules (computed once, shared by every investor scenario)
# ----------------------------------------------------------------------------
def weight_schedule(strategy: str, cov_method: str, Rm: pd.DataFrame, Rd: pd.DataFrame,
                    rf: pd.Series, oos_dates: pd.DatetimeIndex, rebal: set, window: int,
                    cap: float, single_asset: str = None, equity_assets: List[str] = None) -> pd.DataFrame:
    """Target weights at every rebalance date (and the first OOS date), estimated from data
    available up to that date only (rolling window of `window` months)."""
    assets = list(Rm.columns)
    rows = {}
    for d in oos_dates:
        if d != oos_dates[0] and d not in rebal:
            continue
        if strategy == "EQ100":
            w = pd.Series(0.0, index=assets); w[single_asset] = 1.0
        elif strategy == "EQ_EW":
            w = pd.Series(0.0, index=assets); w[equity_assets] = 1.0 / len(equity_assets)
        elif strategy == "EW":
            w = E.w_equal(assets)
        else:
            win = Rm.loc[:d].iloc[-window:]
            if cov_method == "sample":
                cov = E.cov_sample(win)
            elif cov_method == "lw":
                cov = E.cov_ledoit_wolf(win)
            elif cov_method == "dcc":
                dwin = Rd.loc[:d].iloc[-window * 21:]
                try:
                    cov = E.dcc_forecast(dwin)
                except Exception:
                    cov = E.cov_ledoit_wolf(win)
            else:
                raise ValueError(cov_method)
            if strategy == "MinVar":
                w = E.w_minvar(cov, assets, cap)
            elif strategy == "MaxSharpe":
                w = E.w_maxsharpe(win.mean().values, cov, float(rf.loc[:d].iloc[-window:].mean()), assets, cap)
            elif strategy == "RiskParity":
                w = E.w_riskparity(cov, assets)
            else:
                raise ValueError(strategy)
            w.attrs["dr"] = E.diversification_ratio(w, cov)
        rows[d] = w
    return pd.DataFrame(rows).T


def make_weight_fn(sched: pd.DataFrame, dates: pd.DatetimeIndex):
    def fn(t):
        d = dates[t]
        s = sched.loc[:d]
        return s.iloc[-1] if len(s) else sched.iloc[0]
    return fn


def scenario_objects(mode: str, broker: str, slab: float, cfg: dict, regime: str = None,
                     extra_bps: float = 0.0, gf_prices=None):
    cc = cfg["costs"]
    cost = E.CostModel(brokerage=broker, flat_fee=cc["flat_fee"],
                       stt={k: tuple(v) for k, v in cc["stt"].items()},
                       exchange=cc["exchange"], sebi=cc["sebi"], stamp_buy=cc["stamp_buy"],
                       gst=cc["gst"], dp_charge_per_sell=cc["dp_charge_per_sell"],
                       half_spread=cc["half_spread"], extra_bps=extra_bps,
                       frictionless=(mode == "frictionless"))
    tx = cfg["tax"]
    tax = E.TaxEngine(regime="none" if mode != "costs_tax" else (regime or tx["regime"]),
                      slab=slab, cess=tx["cess"], eq_st_rate=tx["eq_st_rate"],
                      eq_lt_rate=tx["eq_lt_rate"], eq_lt_exemption=tx["eq_lt_exemption"],
                      oth_lt_rate=tx["oth_lt_rate"], cii={int(k): v for k, v in tx["cii"].items()},
                      gf_prices=gf_prices)
    integer = mode != "frictionless"
    return cost, tax, integer


def run_one(prices, types, sched, contrib, rebal, mode, broker, slab, cfg, threshold=None,
            regime=None, extra_bps=0.0, gf_prices=None):
    cost, tax, integer = scenario_objects(mode, broker, slab, cfg, regime, extra_bps, gf_prices)
    return E.simulate(prices, types, make_weight_fn(sched, prices.index), rebal, contrib, cost, tax,
                      integer=integer, threshold=threshold, tax_month=cfg["tax"]["payment_month"],
                      sip_single_order=(broker in cfg.get("sip_single_order_brokers", [])) and mode != "frictionless")


# ----------------------------------------------------------------------------
# Random direct-equity portfolios
# ----------------------------------------------------------------------------
def draw_stock_sets(P: pd.DataFrame, elig: pd.DataFrame, start: pd.Timestamp, N: int,
                    draws: int, seed: int) -> List[List[str]]:
    rng = np.random.default_rng(seed + N)
    pool = [c for c in elig.columns if bool(elig.loc[start, c]) and np.isfinite(P.loc[start, c])]
    return [list(rng.choice(pool, size=N, replace=False)) for _ in range(draws)]


def run_stock_portfolio(P_oos: pd.DataFrame, ids: List[str], contrib, rebal, mode, broker, cfg,
                        regime=None, gf_prices=None):
    prices = P_oos[ids]
    types = {i: "stock" for i in ids}
    cost, tax, integer = scenario_objects(mode, broker, 0.2, cfg, regime, 0.0, gf_prices)

    def wfn(t):
        live = [i for i in ids if np.isfinite(prices.iloc[t][i])]
        return pd.Series(1.0 / len(live), index=live) if live else pd.Series(dtype=float)
    return E.simulate(prices, types, wfn, rebal, contrib, cost, tax, integer=integer,
                      tax_month=cfg["tax"]["payment_month"],
                      sip_single_order=(broker in cfg.get("sip_single_order_brokers", [])) and mode != "frictionless")


# ----------------------------------------------------------------------------
# Sub-periods
# ----------------------------------------------------------------------------
def subperiod_table(returns: Dict[str, pd.Series], rf: pd.Series, periods: Dict[str, List[str]]):
    rows = []
    for pname, (a, b) in periods.items():
        for s, r in returns.items():
            rr = r.loc[a:b].dropna()
            if len(rr) < 6:
                continue
            m = E.perf_metrics(rr, rf)
            rows.append({"subperiod": pname, "strategy": s, "months": len(rr),
                         "ann_return": m["ann_return"], "ann_vol": m["ann_vol"],
                         "sharpe": m["sharpe"], "max_drawdown": m["max_drawdown"]})
    return pd.DataFrame(rows)


# ----------------------------------------------------------------------------
# Figures
# ----------------------------------------------------------------------------
def fig_cumulative(ret: Dict[str, pd.Series], band: pd.DataFrame = None, band_label: str = ""):
    fig, ax = plt.subplots(figsize=(7.0, 3.6))
    if band is not None and len(band):
        cum = (1 + band.fillna(0)).cumprod()
        lo, md, hi = cum.quantile(0.05, axis=1), cum.median(axis=1), cum.quantile(0.95, axis=1)
        ax.fill_between(cum.index, lo, hi, color=STOCK_GREY, alpha=0.18, lw=0,
                        label=f"{band_label}: 5th-95th pct")
        ax.plot(md.index, md, color=STOCK_GREY, lw=1.4, ls="--", label=f"{band_label}: median")
    for i, (k, r) in enumerate(ret.items()):
        c = (1 + r.fillna(0)).cumprod()
        ax.plot(c.index, c, color=PALETTE[i % 8], label=k)
    ax.set_ylabel("Growth of Rs 1 (time-weighted, net)")
    ax.legend(ncol=3, loc="upper left")
    return fig


def fig_drawdowns(ret: Dict[str, pd.Series]):
    fig, ax = plt.subplots(figsize=(7.0, 3.0))
    for i, (k, r) in enumerate(ret.items()):
        w = (1 + r.fillna(0)).cumprod()
        ax.plot(w.index, (w / w.cummax() - 1) * 100, color=PALETTE[i % 8], label=k)
    ax.set_ylabel("Drawdown (%)")
    ax.legend(ncol=3, loc="lower left")
    return fig


def fig_corr(Rm: pd.DataFrame, labels: Dict[str, str]):
    C = Rm.corr()
    names = [labels.get(c, c) for c in C.columns]
    fig, ax = plt.subplots(figsize=(4.6, 3.9))
    im = ax.imshow(C.values, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=35, ha="right")
    ax.set_yticks(range(len(names))); ax.set_yticklabels(names)
    ax.grid(False)
    for i in range(len(names)):
        for j in range(len(names)):
            ax.text(j, i, f"{C.values[i, j]:.2f}", ha="center", va="center", fontsize=8,
                    color="white" if abs(C.values[i, j]) > 0.6 else INK)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Correlation")
    return fig


def fig_dcc(dcc: pd.DataFrame, pairs: List[str]):
    fig, ax = plt.subplots(figsize=(7.0, 3.0))
    m = dcc[pairs].resample("ME").mean()
    for i, p in enumerate(pairs):
        ax.plot(m.index, m[p], color=PALETTE[i % 8], label=p)
    ax.axhline(0, color=INK2, lw=0.8)
    ax.set_ylabel("DCC correlation (monthly avg)")
    ax.legend(ncol=2, loc="lower left")
    return fig


def fig_sharpe_dist(stock_sharpes: pd.DataFrame, etf_sharpes: Dict[str, float]):
    """stock_sharpes: long df with columns N, sharpe."""
    Ns = sorted(stock_sharpes["N"].unique())
    fig, ax = plt.subplots(figsize=(7.0, 3.6))
    data = [stock_sharpes.loc[stock_sharpes["N"] == n, "sharpe"].dropna().values for n in Ns]
    parts = ax.violinplot(data, positions=range(len(Ns)), showmedians=True, widths=0.75)
    for b in parts["bodies"]:
        b.set_facecolor(STOCK_GREY); b.set_edgecolor(INK2); b.set_alpha(0.35)
    for k in ("cmedians", "cmins", "cmaxes", "cbars"):
        parts[k].set_color(INK2); parts[k].set_linewidth(1)
    for i, (k, v) in enumerate(etf_sharpes.items()):
        ax.axhline(v, color=PALETTE[i % 8], lw=1.6, label=f"{k} (Sharpe {v:.2f})")
    ax.set_xticks(range(len(Ns))); ax.set_xticklabels([f"{n} stocks" for n in Ns])
    ax.set_ylabel("Annualised Sharpe ratio (net of costs & tax)")
    ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.1), fontsize=7.5)
    return fig


def fig_erosion(df: pd.DataFrame):
    """df columns: investor, portfolio, cost_pct, tax_pct (as % of contributions)."""
    inv = list(dict.fromkeys(df["investor"]))
    ports = list(dict.fromkeys(df["portfolio"]))
    fig, ax = plt.subplots(figsize=(7.0, 3.4))
    w = 0.8 / len(ports)
    for j, p in enumerate(ports):
        sub = df[df["portfolio"] == p].set_index("investor").reindex(inv)
        x = np.arange(len(inv)) + (j - (len(ports) - 1) / 2) * w
        ax.bar(x, sub["cost_pct"], w * 0.92, color=PALETTE[j % 8], label=f"{p}: costs")
        ax.bar(x, sub["tax_pct"], w * 0.92, bottom=sub["cost_pct"], color=PALETTE[j % 8],
               alpha=0.45, hatch="///", edgecolor="white", linewidth=0, label=f"{p}: tax")
    ax.set_xticks(range(len(inv))); ax.set_xticklabels(inv, rotation=20, ha="right")
    ax.set_ylabel("% of total contributions")
    ax.legend(ncol=2)
    return fig


def fig_frontier(Rm: pd.DataFrame, labels: Dict[str, str], points: Dict[str, pd.Series],
                 stock_cloud: pd.DataFrame = None):
    from sklearn.covariance import LedoitWolf
    mu = Rm.mean().values * 12
    cov = LedoitWolf().fit(Rm.values).covariance_ * 12
    n = len(mu)
    from scipy.optimize import minimize
    targets = np.linspace(mu.min(), mu.max(), 40)
    vols = []
    for tg in targets:
        cons = [{"type": "eq", "fun": lambda w: w.sum() - 1},
                {"type": "eq", "fun": lambda w, tg=tg: w @ mu - tg}]
        r = minimize(lambda w: w @ cov @ w, np.full(n, 1 / n), bounds=[(0, 1)] * n,
                     constraints=cons, method="SLSQP")
        vols.append(np.sqrt(r.fun) if r.success else np.nan)
    fig, ax = plt.subplots(figsize=(6.2, 4.0))
    if stock_cloud is not None and len(stock_cloud):
        ax.scatter(stock_cloud["ann_vol"] * 100, stock_cloud["ann_return"] * 100, s=8,
                   color=STOCK_GREY, alpha=0.35, lw=0, label="Random 10-stock portfolios (realised)")
    ax.plot(np.array(vols) * 100, targets * 100, color=INK, lw=1.6, label="Long-only ETF frontier")
    sd = np.sqrt(np.diag(cov))
    for i, c in enumerate(Rm.columns):
        ax.scatter(sd[i] * 100, mu[i] * 100, s=40, marker="s", color=INK2)
        ax.annotate(labels.get(c, c), (sd[i] * 100, mu[i] * 100), xytext=(4, 3),
                    textcoords="offset points", fontsize=7.5, color=INK2)
    for i, (k, w) in enumerate(points.items()):
        w = w.reindex(Rm.columns).fillna(0).values
        ax.scatter(np.sqrt(w @ cov @ w) * 100, w @ mu * 100, s=70, color=PALETTE[(i + 1) % 8],
                   edgecolor="white", linewidth=1.5, zorder=5, label=k)
    ax.set_xlabel("Annualised volatility (%)"); ax.set_ylabel("Annualised mean return (%)")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=3, fontsize=7.5)
    return fig


def fig_rolling_sharpe(ret: Dict[str, pd.Series], rf: pd.Series, win: int = 36):
    fig, ax = plt.subplots(figsize=(7.0, 3.0))
    for i, (k, r) in enumerate(ret.items()):
        ex = r - rf.reindex(r.index).fillna(0)
        rs = ex.rolling(win).mean() / ex.rolling(win).std() * np.sqrt(12)
        ax.plot(rs.index, rs, color=PALETTE[i % 8], label=k)
    ax.axhline(0, color=INK2, lw=0.8)
    ax.set_ylabel(f"Rolling {win}-month Sharpe")
    ax.legend(ncol=3, loc="upper left")
    return fig


def fig_weights(scheds: Dict[str, pd.DataFrame], labels: Dict[str, str]):
    fig, axes = plt.subplots(1, len(scheds), figsize=(3.4 * len(scheds), 2.8), sharey=True)
    axes = np.atleast_1d(axes)
    for ax, (k, s) in zip(axes, scheds.items()):
        cols = list(s.columns)
        ax.stackplot(s.index, (s[cols] * 100).T.values, colors=PALETTE[:len(cols)],
                     labels=[labels.get(c, c) for c in cols], edgecolor="white", linewidth=0.6,
                     step="post")
        ax.set_title(k); ax.set_ylim(0, 100)
        import matplotlib.dates as mdates
        ax.xaxis.set_major_locator(mdates.YearLocator(2))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    axes[0].set_ylabel("Target weight (%)")
    axes[-1].legend(loc="center left", bbox_to_anchor=(1.0, 0.5))
    return fig


def fig_tracking(te: pd.DataFrame):
    fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.8))
    if "TE2_sd_ann" in te:
        s = te["TE2_sd_ann"].dropna() * 100
        axes[0].barh(s.index, s.values, color=PALETTE[0], height=0.6)
        axes[0].set_xlabel("Tracking error, SD of differences (% p.a.)")
    if "pd_mean_abs" in te:
        s = te["pd_mean_abs"].dropna() * 100
        axes[1].barh(s.index, s.values, color=PALETTE[1], height=0.6)
        axes[1].set_xlabel("Mean |premium/discount| to NAV (%)")
    fig.tight_layout()
    return fig
''')
print('wrote src/study.py')

In [ ]:
import importlib
sys.path.insert(0, DIRS["src"])
import etf_engine as E, data_io as D, study as S
for m in (E, D, S):
    importlib.reload(m)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
S.set_style()
OUT = S.Outputs(DIRS)
MAN = D.Manifest(os.path.join(BASE, "data", "download_manifest.jsonl"))

## 4. Configuration
Every assumption lives here and is saved to `config/`. Cost and tax parameters carry their sources.

In [ ]:
CONFIG = {
  "study": {
    "title": "Do Index ETFs Help Small Investors Diversify? Evidence from India with Realistic Costs, Taxes and Whole-Unit Purchases",
    "author": AUTHOR, "run_id": RUN_ID, "data_mode": DATA_MODE, "quick_run": QUICK_RUN},
  "period": {"start": "2015-01-01", "end": "2026-09-30"},
  # One ETF per asset class. Candidates are tried in order; the most liquid one with the
  # longest history passing the liquidity screen is kept (see Table 1).
  "etf_sleeves": {
    "eq_large":  {"label": "Nifty 50",      "type": "eq_etf",   "candidates": ["NIFTYBEES.NS", "SETFNIF50.NS", "NIFTYIETF.NS"], "tri_index": "NIFTY 50",      "min_traded_value_cr": 0.5},
    "eq_next50": {"label": "Nifty Next 50", "type": "eq_etf",   "candidates": ["JUNIORBEES.NS", "SETFNN50.NS"],                "tri_index": "NIFTY NEXT 50", "min_traded_value_cr": 0.2},
    "gold":      {"label": "Gold",          "type": "gold_etf", "candidates": ["GOLDBEES.NS", "HDFCGOLD.NS", "SETFGOLD.NS"],   "tri_index": None,            "min_traded_value_cr": 0.2},
    "debt":      {"label": "G-Sec",         "type": "debt_etf", "candidates": ["LTGILTBEES.NS", "SETF10GILT.NS", "LICNETFGSC.NS", "GILT5YBEES.NS"], "tri_index": None, "min_traded_value_cr": 0.005},
    "intl":      {"label": "Nasdaq 100",    "type": "intl_etf", "candidates": ["MON100.NS", "HNGSNGBEES.NS"],                 "tri_index": None,            "min_traded_value_cr": 0.1},
  },
  # Force a specific ticker for a sleeve (e.g. {"debt": "LTGILTBEES.NS"}) - used for robustness runs.
  "etf_override": {},
  # Return series per sleeve: "price" (market price, default) or "nav" (AMFI NAV, unit-normalised).
  # The G-Sec ETF trades thinly (stale prices, ~28% zero-volume days in the quick run), so its NAV is used;
  # Section 17 reports the market-price version as a robustness check.
  "price_source": {"debt": "nav"},
  # mfapi.in search terms used to find each ETF's NAV history (AMFI data)
  "nav_search": {
    "NIFTYBEES.NS":  {"query": "Nippon India ETF Nifty 50 BeES", "must": ["nifty 50", "bees"]},
    "SETFNIF50.NS":  {"query": "SBI Nifty 50 ETF", "must": ["nifty 50", "etf"]},
    "JUNIORBEES.NS": {"query": "Nippon India ETF Nifty Next 50 Junior BeES", "must": ["next 50", "bees"]},
    "SETFNN50.NS":   {"query": "SBI Nifty Next 50 ETF", "must": ["next 50", "etf"]},
    "GOLDBEES.NS":   {"query": "Nippon India ETF Gold BeES", "must": ["gold", "bees"]},
    "HDFCGOLD.NS":   {"query": "HDFC Gold ETF", "must": ["gold", "etf"]},
    "SETFGOLD.NS":   {"query": "SBI Gold ETF", "must": ["gold", "etf"]},
    "LTGILTBEES.NS": {"query": "Nippon India ETF Nifty 8-13 yr G-Sec Long Term Gilt", "must": ["gilt"]},
    "SETF10GILT.NS": {"query": "SBI Nifty 10 yr Benchmark G-Sec ETF", "must": ["g-sec", "etf"]},
    "LICNETFGSC.NS": {"query": "LIC MF Nifty 8-13 yr G-Sec ETF", "must": ["g-sec", "etf"]},
    "GILT5YBEES.NS": {"query": "Nippon India ETF Nifty 5 yr Benchmark G-Sec", "must": ["5 yr", "g-sec"]},
    "MON100.NS":     {"query": "Motilal Oswal Nasdaq 100 ETF", "must": ["nasdaq 100", "etf"]},
    "HNGSNGBEES.NS": {"query": "Nippon India ETF Hang Seng BeES", "must": ["hang seng", "bees"]},
  },
  "risk_free": {"fred_series": "INTGSTINM193N",
                "fallback_liquid_fund": {"query": "Nippon India Liquid Fund", "must": ["liquid", "direct", "growth"]}},
  "clean_spikes": True,     # fix bad ticks in ETF daily prices/NAVs and one-month spike-reversals in stock prices
  "stocks": {"large_cap_top_n": 100, "top_n": 500, "min_price": 10.0, "lookback_months": 3, "min_yahoo_coverage": 0.9,
             "N_list": [5, 10, 15, 20], "draws": 40 if QUICK_RUN else 500,
             "draws_grid": 15 if QUICK_RUN else 100, "seed": 2026},
  "backtest": {"window": 36, "rebalance": "A", "weight_cap": 0.40,
               "strategies": ["EQ100", "EQ_EW", "EW", "MinVar", "MaxSharpe", "RiskParity"], "main_cov": "lw",
               "cov_methods": ["sample", "lw", "dcc"], "gamma": 1.0},
  "labels": {"EQ100": "Nifty 50 ETF only", "EQ_EW": "Equity ETFs 1/N", "EW": "1/N ETF", "MinVar": "Min-variance ETF",
             "MaxSharpe": "Max-Sharpe ETF", "RiskParity": "Risk-parity ETF"},
  "investors": {"lump": [10000, 50000, 100000], "sip": [1000, 5000, 10000]},
  "main_investor": {"mode": "sip", "amount": 5000},
  "main_lump": {"mode": "lump", "amount": 50000},
  "brokers": ["zero", "flat"], "main_broker": "zero",
  # With a flat fee per order, a small SIP investor puts each month's instalment into the single most
  # underweight holding (one order) instead of buying every holding; full rebalancing still trades all.
  "sip_single_order_brokers": ["flat"],
  "slabs": [0.05, 0.20, 0.30], "main_slab": 0.20,
  "costs": {
    "flat_fee": 20.0,
    "stt": {"stock": [0.001, 0.001], "eq_etf": [0.0, 0.00001], "gold_etf": [0.0, 0.0],
            "debt_etf": [0.0, 0.0], "intl_etf": [0.0, 0.0]},
    "exchange": 0.0000297, "sebi": 0.000001, "stamp_buy": 0.00015, "gst": 0.18,
    "dp_charge_per_sell": 15.93,
    "half_spread": {"stock": 0.0010, "eq_etf": 0.0005, "gold_etf": 0.0010, "debt_etf": 0.0015, "intl_etf": 0.0020},
    "notes": {
      "stt": "Delivery equity 0.1% buy and sell; equity ETF units 0.001% on sale only; non-equity ETFs nil. NSE: https://www.nseindia.com/static/invest/first-time-investor-sebi-turnover-fees-stt-other-levies",
      "stamp_buy": "0.015% on delivery purchases, paid by buyer (Indian Stamp Act amendment, 1 July 2020). Source as above.",
      "exchange": "NSE cash-market transaction charge 0.00297% (uniform rate). Verify on the NSE charges page before submission.",
      "sebi": "SEBI turnover fee Rs 10 per crore.",
      "gst": "18% on brokerage, exchange and SEBI fees.",
      "dp_charge_per_sell": "Depository + broker DP charge per scrip per sell day, incl. GST (typical discount broker). Verify for the broker you describe.",
      "flat_fee": "Rs 20 per executed order (typical discount broker intraday/F&O fee; several brokers charge it on delivery too).",
      "half_spread": "Assumed half bid-ask spread; robustness adds 10/25/50 bps."}},
  "tax": {
    "regime": "current", "cess": 0.04, "eq_st_rate": 0.20, "eq_lt_rate": 0.125, "eq_lt_exemption": 125000.0,
    "oth_lt_rate": 0.125, "payment_month": 7,
    "cii": {2014: 240, 2015: 254, 2016: 264, 2017: 272, 2018: 280, 2019: 289, 2020: 301, 2021: 317,
            2022: 331, 2023: 348, 2024: 363, 2025: 376},
    "notes": {
      "equity": "Stocks and equity ETFs: STCG 20% (Sec 111A) if held <= 12 months; LTCG 12.5% above Rs 1.25 lakh per FY aggregate (Sec 112A). ClearTax: https://cleartax.in/s/how-are-etfs-taxed",
      "gold_intl": "Gold and international ETFs: STCG at slab rate (<= 12 months); LTCG 12.5% without indexation, no Rs 1.25 lakh exemption. IIFL: https://www.iifl.com/blogs/gold-loan/gold-etf-tax-in-india-2026-post-finance-act-2024-rules",
      "debt": "Debt ETFs bought on/after 1 April 2023: slab rate regardless of holding period (Sec 50AA). Finnovate: https://www.finnovate.in/learn/blog/etf-taxation-india",
      "unchanged_2026": "Structure set by Finance (No.2) Act 2024 and unchanged in Budget 2026. https://finlecture.in/indian-tax-system/capital-gains-tax-fy-2026-27/",
      "historical_regime": "Robustness only: equity LTCG exempt before 1-Apr-2018, 10% above Rs 1 lakh until 22-Jul-2024 with 31-Jan-2018 grandfathering; equity STCG 15% until 22-Jul-2024; non-equity LTCG 20% with indexation after 36 months for sales before 23-Jul-2024; Sec 50AA slab taxation for units bought from 1-Apr-2023 (gold/international until 31-Mar-2025). CII values should be verified against CBDT notifications.",
      "simplifications": "Resident individual; 4% cess; no surcharge; tax paid each July from the portfolio; losses set off and carried forward; dividends ignored (growth ETFs)."}},
  "tests": {"bootstrap_M": 499 if QUICK_RUN else 4999, "stock_test_M": 199 if QUICK_RUN else 999,
            "block": 5, "alpha": 0.05, "seed": 42},
  "robustness": {"windows": [36, 48, 60], "rebalance": ["M", "Q", "A", "T5"], "extra_bps": [10, 25, 50]},
  "subperiods": {"Pre-COVID": ["2015-01", "2020-01"], "COVID crash and recovery": ["2020-02", "2021-12"],
                 "2022 rate hikes": ["2022-01", "2022-12"], "2023-2024": ["2023-01", "2024-12"],
                 "2025 tariff shock": ["2025-01", "2025-12"], "2026 to date": ["2026-01", "2026-12"]},
}
with open(f"{DIRS['config']}/config.json", "w") as f: json.dump(CONFIG, f, indent=2)
with open(f"{DIRS['config']}/config_{RUN_ID}.json", "w") as f: json.dump(CONFIG, f, indent=2)
with open(f"{DIRS['logs']}/config_used.json", "w") as f: json.dump(CONFIG, f, indent=2)
LBL = CONFIG["labels"]
print("Configuration saved.")

## 5. References and documentation
Writes the bibliography, data-source register, cost/tax rule sheet, assumptions log and README.

In [ ]:
TODAY = dt.date.today().isoformat()
BIB = r"""
@article{markowitz1952, author={Markowitz, Harry}, title={Portfolio Selection}, journal={Journal of Finance}, volume={7}, number={1}, pages={77--91}, year={1952}}
@article{tobin1958, author={Tobin, James}, title={Liquidity Preference as Behavior towards Risk}, journal={Review of Economic Studies}, volume={25}, number={2}, pages={65--86}, year={1958}}
@article{sharpe1964, author={Sharpe, William F.}, title={Capital Asset Prices: A Theory of Market Equilibrium under Conditions of Risk}, journal={Journal of Finance}, volume={19}, number={3}, pages={425--442}, year={1964}}
@article{evansarcher1968, author={Evans, John L. and Archer, Stephen H.}, title={Diversification and the Reduction of Dispersion: An Empirical Analysis}, journal={Journal of Finance}, volume={23}, number={5}, pages={761--767}, year={1968}}
@article{fama1970, author={Fama, Eugene F.}, title={Efficient Capital Markets: A Review of Theory and Empirical Work}, journal={Journal of Finance}, volume={25}, number={2}, pages={383--417}, year={1970}}
@article{samuelson1974, author={Samuelson, Paul A.}, title={Challenge to Judgment}, journal={Journal of Portfolio Management}, volume={1}, number={1}, pages={17--19}, year={1974}}
@article{brinson1986, author={Brinson, Gary P. and Hood, L. Randolph and Beebower, Gilbert L.}, title={Determinants of Portfolio Performance}, journal={Financial Analysts Journal}, volume={42}, number={4}, pages={39--44}, year={1986}}
@article{statman1987, author={Statman, Meir}, title={How Many Stocks Make a Diversified Portfolio?}, journal={Journal of Financial and Quantitative Analysis}, volume={22}, number={3}, pages={353--363}, year={1987}}
@article{sharpe1991, author={Sharpe, William F.}, title={The Arithmetic of Active Management}, journal={Financial Analysts Journal}, volume={47}, number={1}, pages={7--9}, year={1991}}
@article{barberodean2000, author={Barber, Brad M. and Odean, Terrance}, title={Trading Is Hazardous to Your Wealth: The Common Stock Investment Performance of Individual Investors}, journal={Journal of Finance}, volume={55}, number={2}, pages={773--806}, year={2000}}
@article{engle2002, author={Engle, Robert}, title={Dynamic Conditional Correlation: A Simple Class of Multivariate GARCH Models}, journal={Journal of Business and Economic Statistics}, volume={20}, number={3}, pages={339--350}, year={2002}}
@article{ledoitwolf2004, author={Ledoit, Olivier and Wolf, Michael}, title={Honey, I Shrunk the Sample Covariance Matrix}, journal={Journal of Portfolio Management}, volume={30}, number={4}, pages={110--119}, year={2004}}
@article{romanowolf2005, author={Romano, Joseph P. and Wolf, Michael}, title={Stepwise Multiple Testing as Formalized Data Snooping}, journal={Econometrica}, volume={73}, number={4}, pages={1237--1282}, year={2005}}
@article{french2008, author={French, Kenneth R.}, title={The Cost of Active Investing}, journal={Journal of Finance}, volume={63}, number={4}, pages={1537--1573}, year={2008}}
@article{goetzmannkumar2008, author={Goetzmann, William N. and Kumar, Alok}, title={Equity Portfolio Diversification}, journal={Review of Finance}, volume={12}, number={3}, pages={433--463}, year={2008}}
@article{ledoitwolf2008, author={Ledoit, Olivier and Wolf, Michael}, title={Robust Performance Hypothesis Testing with the Sharpe Ratio}, journal={Journal of Empirical Finance}, volume={15}, number={5}, pages={850--859}, year={2008}}
@article{demiguel2009, author={DeMiguel, Victor and Garlappi, Lorenzo and Uppal, Raman}, title={Optimal Versus Naive Diversification: How Inefficient is the 1/N Portfolio Strategy?}, journal={Review of Financial Studies}, volume={22}, number={5}, pages={1915--1953}, year={2009}}
@article{ledoitwolf2011, author={Ledoit, Olivier and Wolf, Michael}, title={Robust Performance Hypothesis Testing with the Variance}, journal={Wilmott}, volume={2011}, number={55}, pages={86--89}, year={2011}}
@article{baumanntrautmann2013, author={Baumann, Philipp and Trautmann, Norbert}, title={Portfolio-Optimization Models for Small Investors}, journal={Mathematical Methods of Operations Research}, volume={77}, number={3}, pages={345--356}, year={2013}}
@article{andrews1991, author={Andrews, Donald W. K.}, title={Heteroskedasticity and Autocorrelation Consistent Covariance Matrix Estimation}, journal={Econometrica}, volume={59}, number={3}, pages={817--858}, year={1991}}
@article{singhkaur2016, author={Singh, Jaspal and Kaur, Parneet}, title={Tracking Efficiency of Exchange Traded Funds (ETFs): Empirical Evidence from Indian Equity ETFs}, journal={Paradigm}, volume={20}, number={2}, pages={176--190}, year={2016}}
@article{tripathisethi2022, author={Tripathi, Vanita and Sethi, Aakanksha}, title={An Evaluation of the Tracking Performance of Exchange Traded Funds (ETFs): The Case of Indian Index ETFs}, journal={Vision: The Journal of Business Perspective}, year={2022}, doi={10.1177/0972262921996485}}
@techreport{rajuagarwalla2021, author={Raju, Rajan and Agarwalla, Sobhesh Kumar}, title={Equity Portfolio Diversification: How Many Stocks Are Enough? Evidence from India}, institution={Indian Institute of Management Ahmedabad}, type={Working Paper}, year={2021}, url={https://www.iima.ac.in/publication/equity-portfolio-diversification-how-many-stocks-are-enough-evidence-india}}
@misc{spivaindia2025, author={{S\&P Dow Jones Indices}}, title={SPIVA India Scorecard Year-End 2025}, year={2026}, url={https://www.spglobal.com/spdji/en/spiva/article/spiva-india-year-end-2025/}}
@misc{data_yahoo, author={{Yahoo Finance}}, title={Historical prices via the yfinance Python package}, url={https://pypi.org/project/yfinance/}, note={Accessed TODAY}}
@misc{data_nse_bhav, author={{National Stock Exchange of India}}, title={Capital Market Bhavcopy Archives}, url={https://www.nseindia.com/all-reports}, note={Accessed TODAY}}
@misc{data_amfi, author={{Association of Mutual Funds in India}}, title={NAV history (via mfapi.in)}, url={https://www.mfapi.in/}, note={Accessed TODAY}}
@misc{data_fred, author={{Federal Reserve Bank of St. Louis}}, title={Interest Rates, Government Securities, Treasury Bills for India (INTGSTINM193N)}, url={https://fred.stlouisfed.org/series/INTGSTINM193N}, note={Accessed TODAY}}
@misc{data_niftyindices, author={{NSE Indices Ltd}}, title={Total Return Index historical data}, url={https://www.niftyindices.com/reports/historical-data}, note={Accessed TODAY}}
@misc{nse_levies, author={{National Stock Exchange of India}}, title={Turnover fees, STT and other levies}, url={https://www.nseindia.com/static/invest/first-time-investor-sebi-turnover-fees-stt-other-levies}, note={Accessed TODAY}}
@misc{cleartax_etf, author={{ClearTax}}, title={How ETFs Are Taxed in India (FY 2025-26)}, url={https://cleartax.in/s/how-are-etfs-taxed}, note={Accessed TODAY}}
""".replace("TODAY", TODAY)
open(f"{DIRS['references']}/references.bib", "w").write(BIB.strip() + "\n")

open(f"{DIRS['references']}/data_sources.md", "w").write(f"""# Data sources (all free)

| Data | Source | Access | Use in study |
|---|---|---|---|
| ETF daily prices, volume | Yahoo Finance (`yfinance`) | Python package | Tradable prices, returns, liquidity |
| ETF daily NAV | AMFI NAV history via https://api.mfapi.in | JSON API | Tracking error, premium/discount |
| Index total return (TRI) | https://www.niftyindices.com (historical data) | JSON endpoint | Tracking error benchmark |
| All listed equities, month-end | NSE bhavcopy archives (nsearchives.nseindia.com) | ZIP files | Survivorship-free stock universe |
| Stock adjusted prices | Yahoo Finance; bhavcopy closes with split/bonus correction as fallback | | Random stock portfolios |
| Risk-free rate | FRED INTGSTINM193N (India 3-month T-bill); liquid-fund NAV return as fallback | CSV | Excess returns, Sharpe |

Every downloaded file is logged in `data/download_manifest.jsonl` with URL, UTC time, SHA-256 and row count.
First written {TODAY}.
""")

open(f"{DIRS['references']}/tax_cost_rules.md", "w").write(
    "# Transaction-cost and tax rules used\n\nGenerated from config.json on " + TODAY + ".\n\n## Costs\n\n" +
    "\n".join(f"- **{k}**: {v}" for k, v in CONFIG["costs"]["notes"].items()) +
    "\n\n## Taxes\n\n" + "\n".join(f"- **{k}**: {v}" for k, v in CONFIG["tax"]["notes"].items()) +
    "\n\nCII (base 2001-02 = 100): " + json.dumps(CONFIG["tax"]["cii"]) + "\n")

open(f"{DIRS['references']}/methodology_assumptions.md", "w").write(f"""# Methodology and assumptions log

1. **Out-of-sample design.** Weights are estimated on a rolling {CONFIG['backtest']['window']}-month window using data up to the trade date only (DeMiguel et al., 2009). Trades execute at month-end closing prices.
2. **ETF strategies.** Nifty 50 ETF only; 1/N across ETFs; minimum-variance; constrained maximum-Sharpe; risk parity. Long-only, weight cap {CONFIG['backtest']['weight_cap']:.0%} for optimised portfolios. Covariance: Ledoit-Wolf shrinkage (main), sample and DCC-GARCH (robustness).
3. **Direct-equity portfolios.** N in {CONFIG['stocks']['N_list']} stocks drawn at random from the {CONFIG['stocks']['top_n']} most-traded NSE equities on the start date (3-month average traded value, price at least Rs {CONFIG['stocks']['min_price']:.0f}). Universe built from month-end bhavcopies, so later-delisted stocks are included. Equal weights, held to the end; delisted stocks sold at their last price and proceeds reinvested at the next trade.
4. **Small-investor frictions.** Whole units only; greedy allocation of available cash toward target weights; leftover cash earns zero. Costs: brokerage (zero or Rs 20 flat), STT, exchange, SEBI, stamp duty, GST, DP charge and half-spread.
5. **SIP.** Monthly contribution buys toward target weights without selling; full rebalancing annually (main case).
6. **Tax.** FIFO lots; FY (April-March) netting, loss set-off and carry-forward; tax paid each July from the portfolio; terminal liquidation taxed. Current FY2026-27 rules in the main case, historical date-dependent rules as robustness.
7. **Returns.** Time-weighted monthly returns net of costs and taxes paid; money-weighted XIRR on post-tax terminal wealth.
8. **Inference.** Ledoit-Wolf (2008) studentised circular block bootstrap for Sharpe differences, Ledoit-Wolf (2011) for variance differences, Romano-Wolf (2005) stepdown for multiple comparisons; block length {CONFIG['tests']['block']}.
9. **Known limitations.** Adjusted prices used as tradable prices; dividends on stocks reinvested via adjusted prices and not taxed separately; delisting returns set to last traded price; Yahoo coverage of delisted stocks is incomplete (bhavcopy fallback with split heuristic, share reported in the stock-universe table); expense ratios captured through ETF prices; ETF AUM not used in screening (liquidity screen on traded value instead).
""")

if not os.path.exists(f"{BASE}/README.md"):
    open(f"{BASE}/README.md", "w").write(f"""# {CONFIG['study']['title']}

Author: {AUTHOR or '(add name)'}

Reproduce: open `ETF_SmallInvestor_India.ipynb` in Google Colab and run all cells.
Raw downloads are cached in `data/raw/`; each run writes to `results/run_<timestamp>/`.

- `config/` every run's parameters
- `references/` bibliography (BibTeX), data sources, tax and cost rules, assumptions
- `data/download_manifest.jsonl` provenance of every downloaded file
- `results/run_*/tables` CSV + LaTeX + all_tables.xlsx; `figures` PNG (300 dpi) + PDF; `key_results.md`
""")
print("References and documentation written to", DIRS["references"])

## 6. Data acquisition
Live mode downloads from free sources and caches raw files; synthetic mode generates placeholder data with the same structure.

In [ ]:
P_DIR = DIRS["processed"]
cfg_p = CONFIG["period"]
SLEEVES = CONFIG["etf_sleeves"]

def cached(path):
    return os.path.exists(path) and not FORCE_DOWNLOAD

if DATA_MODE == "synthetic":
    syn = D.synthetic_dataset(list(SLEEVES), cfg_p["start"], cfg_p["end"])
    syn["etf_daily"].to_csv(f"{P_DIR}/etf_daily.csv.gz", index=False)
    syn["etf_nav"].to_csv(f"{P_DIR}/etf_nav.csv.gz", index=False)
    syn["etf_tri"].to_csv(f"{P_DIR}/etf_tri.csv.gz", index=False)
    syn["rf"].assign(source="synthetic").to_csv(f"{P_DIR}/rf_monthly.csv", index=False)
    syn["stock_prices"].to_csv(f"{P_DIR}/stock_prices_monthly.csv.gz")
    syn["stock_eligible"].to_csv(f"{P_DIR}/stock_eligible.csv.gz")
    syn["stock_rank"].to_csv(f"{P_DIR}/stock_rank.csv.gz")
    syn["stock_meta"].to_csv(f"{P_DIR}/stock_meta.csv", index=False)
    pd.DataFrame([{"sleeve": k, "label": v["label"], "ticker": f"SYN_{k}", "type": v["type"]} for k, v in SLEEVES.items()]
                 ).to_csv(f"{P_DIR}/etf_selection.csv", index=False)
    print("Synthetic data generated (smoke test only).")

In [ ]:
# ---- 6a. ETF prices (Yahoo Finance) and selection of one ETF per sleeve --------------------
if DATA_MODE == "live":
    raw_path = f"{DIRS['raw_etf']}/etf_candidates_daily.csv.gz"
    tickers = sorted({t for s in SLEEVES.values() for t in s["candidates"]})
    if cached(raw_path):
        etf_raw = pd.read_csv(raw_path, parse_dates=["date"])
    else:
        etf_raw = D.yf_history(tickers, cfg_p["start"], (pd.Timestamp(cfg_p["end"]) + pd.Timedelta(days=1)).strftime("%Y-%m-%d"))
        etf_raw.to_csv(raw_path, index=False)
        MAN.log("Yahoo Finance", "yfinance:" + ",".join(tickers), raw_path, len(etf_raw))
    rows = []
    for key, s in SLEEVES.items():
        for rank_, tk in enumerate(s["candidates"]):
            d = etf_raw[etf_raw["ticker"] == tk].dropna(subset=["close"])
            if d.empty:
                rows.append({"sleeve": key, "ticker": tk, "available": False}); continue
            d = d[d["volume"] > 0]
            last2y = d[d["date"] >= d["date"].max() - pd.Timedelta(days=730)]
            tv = (last2y["close"] * last2y["volume"]).median() / 1e7
            rows.append({"sleeve": key, "label": s["label"], "ticker": tk, "type": s["type"], "available": True,
                         "first_date": d["date"].min(), "last_date": d["date"].max(),
                         "median_daily_traded_value_cr_2y": tv, "passes_liquidity": tv >= s["min_traded_value_cr"],
                         "candidate_order": rank_})
    cand = pd.DataFrame(rows)
    cand.to_csv(f"{P_DIR}/etf_candidates.csv", index=False)
    ok = cand[cand["available"] & cand["passes_liquidity"]].sort_values(["sleeve", "first_date", "candidate_order"])
    sel = ok.groupby("sleeve").head(1)
    for slv, tk in CONFIG.get("etf_override", {}).items():
        forced = cand[(cand["sleeve"] == slv) & (cand["ticker"] == tk) & cand["available"]]
        if len(forced):
            sel = pd.concat([sel[sel["sleeve"] != slv], forced]).sort_values("sleeve")
            print(f"Override: {slv} -> {tk}")
    missing = set(SLEEVES) - set(sel["sleeve"])
    if missing:
        print("WARNING: no liquid ETF found for", missing, "- check tickers in CONFIG; these sleeves are dropped.")
    sel.to_csv(f"{P_DIR}/etf_selection.csv", index=False)
    etf_daily = etf_raw.merge(sel[["ticker", "sleeve"]], on="ticker").rename(columns={"sleeve": "key"})
    etf_daily[["date", "key", "close", "adj_close", "volume"]].to_csv(f"{P_DIR}/etf_daily.csv.gz", index=False)
    display(cand)

In [ ]:
# ---- 6b. ETF NAVs (AMFI via mfapi) and index TRI (niftyindices) ----------------------------
if DATA_MODE == "live":
    sel = pd.read_csv(f"{P_DIR}/etf_selection.csv")
    navs, maps, tris = [], [], []
    for _, r in sel.iterrows():
        f = f"{DIRS['raw_nav']}/{r['ticker']}.csv"
        spec = CONFIG["nav_search"].get(r["ticker"])
        if cached(f):
            nav = pd.read_csv(f, parse_dates=["date"]); maps.append({"ticker": r["ticker"], "scheme": "cached"})
        elif spec:
            item = D.pick_scheme(spec["query"], spec["must"], avoid=["fund of fund", "fof"])
            if item:
                nav = D.mfapi_nav(item["schemeCode"]); nav.to_csv(f, index=False)
                MAN.log("AMFI via mfapi", f"https://api.mfapi.in/mf/{item['schemeCode']}", f, len(nav), item["schemeName"])
                maps.append({"ticker": r["ticker"], "scheme_code": item["schemeCode"], "scheme": item["schemeName"]})
            else:
                nav = pd.DataFrame(columns=["date", "nav"]); maps.append({"ticker": r["ticker"], "scheme": "NOT FOUND"})
        else:
            nav = pd.DataFrame(columns=["date", "nav"])
        navs.append(nav.assign(key=r["sleeve"]))
        idx = SLEEVES[r["sleeve"]]["tri_index"]
        if idx:
            ft = f"{DIRS['raw_tri']}/{idx.replace(' ', '_')}_api.csv"
            tri = D.read_tri_files(DIRS["raw_tri"], idx)          # manual downloads (and earlier API cache)
            if len(tri):
                print(f"  TRI {idx}: {len(tri)} days from files in data/raw/index_tri ({tri['date'].min():%Y-%m-%d} to {tri['date'].max():%Y-%m-%d})")
            else:
                tri = D.nifty_tri(idx, cfg_p["start"], cfg_p["end"])
                if len(tri):
                    tri.to_csv(ft, index=False); MAN.log("niftyindices TRI", "niftyindices.com", ft, len(tri), idx)
                else:
                    print(f"  TRI for {idx} not available. To add it: niftyindices.com -> Reports -> Historical Data -> "
                          f"Total Returns Index, select {idx}, download each year as CSV and save to "
                          f"{DIRS['raw_tri']}/{idx.replace(' ', '_')}_<year>.csv, then re-run this cell.")
            tris.append(tri.assign(key=r["sleeve"]))
    pd.concat(navs).to_csv(f"{P_DIR}/etf_nav.csv.gz", index=False)
    (pd.concat(tris) if tris else pd.DataFrame(columns=["date", "tri", "key"])).to_csv(f"{P_DIR}/etf_tri.csv.gz", index=False)
    pd.DataFrame(maps).to_csv(f"{P_DIR}/etf_nav_scheme_map.csv", index=False)
    display(pd.DataFrame(maps))

In [ ]:
# ---- 6b-check. NAV coverage, price/NAV consistency and ETF trading activity --------------
if DATA_MODE == "live":
    sel = pd.read_csv(f"{P_DIR}/etf_selection.csv")
    ed = pd.read_csv(f"{P_DIR}/etf_daily.csv.gz", parse_dates=["date"])
    nv = pd.read_csv(f"{P_DIR}/etf_nav.csv.gz", parse_dates=["date"])
    chk = []
    for _, r in sel.iterrows():
        k = r["sleeve"]
        p = ed[ed["key"] == k].set_index("date")
        n_raw = nv[nv["key"] == k].set_index("date")["nav"]
        n = D.normalize_nav_units(p["adj_close"], n_raw)
        fixed_days = int((np.abs(n.reindex(n_raw.index) / n_raw - 1) > 1e-9).sum())
        ratio = (p["close"] / n.reindex(p.index)).dropna()
        yr = ratio.groupby(ratio.index.year).median()
        zero = (p["volume"] == 0).groupby(p.index.year).mean()
        chk.append({"sleeve": k, "ticker": r["ticker"],
                    "nav_first": n.index.min(), "nav_last": n.index.max(), "nav_days": len(n),
                    "price_days": len(p), "nav_coverage_of_price_days": len(ratio) / max(len(p), 1),
                    "NAV days unit-adjusted (split)": fixed_days,
                    "price/NAV median (min year)": yr.min() if len(yr) else np.nan,
                    "price/NAV median (max year)": yr.max() if len(yr) else np.nan,
                    "zero-volume days, worst year": zero.max(), "worst year": zero.idxmax() if len(zero) else None})
    CHECK = pd.DataFrame(chk).set_index("sleeve")
    CHECK.to_csv(f"{P_DIR}/etf_data_check.csv")
    display(CHECK)
    print("What to look for:\n"
          " - nav_coverage close to 1 and nav_first near the ETF's price history start; otherwise the mfapi match is wrong\n"
          "   (fix the search terms in CONFIG['nav_search'] or delete data/raw/etf_nav/<ticker>.csv and re-run 6b).\n"
          " - price/NAV medians (after the automatic unit-split fix) close to 1.00 in every year.\n"
          " - zero-volume share above ~10% in any year means stale prices; consider CONFIG['etf_override'].")

In [ ]:
# ---- 6c. Risk-free rate ------------------------------------------------------------------
if DATA_MODE == "live":
    f = f"{DIRS['raw_rf']}/fred_{CONFIG['risk_free']['fred_series']}.csv"
    if cached(f):
        fr = pd.read_csv(f, parse_dates=["date"])
    else:
        fr = D.fred_series(CONFIG["risk_free"]["fred_series"])
        if len(fr): fr.to_csv(f, index=False); MAN.log("FRED", CONFIG["risk_free"]["fred_series"], f, len(fr))
    months = pd.date_range(cfg_p["start"], cfg_p["end"], freq="ME")
    rf = pd.Series(np.nan, index=months)
    if len(fr):
        m = fr.set_index("date")["value"].resample("ME").last()
        rf = ((1 + m / 100) ** (1 / 12) - 1).reindex(months)
    src = pd.Series(np.where(rf.notna(), "FRED T-bill", "liquid fund NAV"), index=months)
    if rf.isna().any():
        f2 = f"{DIRS['raw_rf']}/liquid_fund_nav.csv"
        if cached(f2):
            lf = pd.read_csv(f2, parse_dates=["date"])
        else:
            spec = CONFIG["risk_free"]["fallback_liquid_fund"]
            item = D.pick_scheme(spec["query"], spec["must"])
            lf = D.mfapi_nav(item["schemeCode"]) if item else pd.DataFrame(columns=["date", "nav"])
            lf.to_csv(f2, index=False); MAN.log("AMFI via mfapi", "liquid fund", f2, len(lf), item["schemeName"] if item else "")
        lr = lf.set_index("date")["nav"].resample("ME").last().pct_change().reindex(months)
        rf = rf.fillna(lr)
    rf = rf.ffill().bfill()
    pd.DataFrame({"date": months, "rf": rf.values, "source": src.values}).to_csv(f"{P_DIR}/rf_monthly.csv", index=False)
    print(src.value_counts())

In [ ]:
# ---- 6d. Survivorship-free stock universe: NSE month-end bhavcopies + Yahoo adjusted prices --
if DATA_MODE == "live":
    bhav = D.month_end_bhavcopies(cfg_p["start"], cfg_p["end"], DIRS["raw_bhav"], MAN, force=FORCE_DOWNLOAD)
    if bhav.empty:
        raise RuntimeError("NSE archives unreachable from this runtime. Re-run later, or use "
                           "D.nifty500_current_list() as a (survivorship-biased) fallback and state it in the paper.")
    b = bhav[bhav["series"] == "EQ"].copy()
    b["id"] = b["isin"].fillna(b["symbol"])
    syms = sorted(b.sort_values("date").groupby("id")["symbol"].last().unique())
    f = f"{DIRS['raw_stock']}/yahoo_stocks_daily.csv.gz"
    if cached(f):
        ys = pd.read_csv(f, parse_dates=["date"])
    else:
        print(f"Downloading {len(syms)} stocks from Yahoo (several minutes)...")
        ys = D.yf_history([s + ".NS" for s in syms], cfg_p["start"],
                          (pd.Timestamp(cfg_p["end"]) + pd.Timedelta(days=1)).strftime("%Y-%m-%d"), batch=100)
        ys[["date", "ticker", "close", "adj_close", "volume"]].to_csv(f, index=False)
        MAN.log("Yahoo Finance", f"yfinance: {len(syms)} NSE stocks", f, len(ys))
    ym = ys.pivot_table(index="date", columns="ticker", values="adj_close").resample("ME").last()
    sc = CONFIG["stocks"]
    P, ELIG, META = D.build_stock_panel(bhav, ym, top_n=sc["top_n"], min_price=sc["min_price"],
                                        lookback=sc["lookback_months"], min_yf_coverage=sc["min_yahoo_coverage"])
    P.to_csv(f"{P_DIR}/stock_prices_monthly.csv.gz"); ELIG.to_csv(f"{P_DIR}/stock_eligible.csv.gz")
    D.LAST_RANK.to_csv(f"{P_DIR}/stock_rank.csv.gz")
    META.to_csv(f"{P_DIR}/stock_meta.csv", index=False)
    print(META["price_source"].value_counts())

## 7. Data preparation

In [ ]:
etf_daily = pd.read_csv(f"{P_DIR}/etf_daily.csv.gz", parse_dates=["date"])
etf_nav = pd.read_csv(f"{P_DIR}/etf_nav.csv.gz", parse_dates=["date"])
etf_tri = pd.read_csv(f"{P_DIR}/etf_tri.csv.gz", parse_dates=["date"])
SEL = pd.read_csv(f"{P_DIR}/etf_selection.csv")
rf_df = pd.read_csv(f"{P_DIR}/rf_monthly.csv", parse_dates=["date"])
RF = rf_df.set_index("date")["rf"]; RF.index = RF.index.to_period("M").to_timestamp("M")
SP = pd.read_csv(f"{P_DIR}/stock_prices_monthly.csv.gz", index_col=0, parse_dates=True)
SE = pd.read_csv(f"{P_DIR}/stock_eligible.csv.gz", index_col=0, parse_dates=True).astype(bool)
SMETA = pd.read_csv(f"{P_DIR}/stock_meta.csv")
_rk = f"{P_DIR}/stock_rank.csv.gz"
SRANK = pd.read_csv(_rk, index_col=0, parse_dates=True) if os.path.exists(_rk) else None
if SRANK is not None:
    SRANK.index = SRANK.index.to_period("M").to_timestamp("M")
SP_RAW = SP.copy()
N_SPIKES = 0
if CONFIG.get("clean_spikes"):
    SP, N_SPIKES = D.clean_monthly_spikes(SP)
    print(f"Stock prices: {N_SPIKES} one-month spike-reversals fixed")
for X in (SP, SP_RAW, SE):
    X.index = X.index.to_period("M").to_timestamp("M")

KEYS = [k for k in SLEEVES if k in set(SEL["sleeve"])]
TYPES = {k: SLEEVES[k]["type"] for k in KEYS}
NAMES = {k: SLEEVES[k]["label"] for k in KEYS}
PXd = S.wide_daily(etf_daily, "key", "adj_close")[KEYS]
ETF_FIXED = {}
if CONFIG.get("clean_spikes"):
    for k in KEYS:
        PXd[k], ETF_FIXED[k] = D.despike_daily(PXd[k])
    print("ETF daily prices, bad ticks fixed:", ETF_FIXED)
PXd_MKT = PXd.copy()                       # market prices, kept for robustness
_navw = S.wide_daily(etf_nav, "key", "nav") if len(etf_nav) else pd.DataFrame()
NAVN = pd.DataFrame({k: D.normalize_nav_units(PXd[k], _navw[k].dropna()) for k in KEYS if k in _navw})
if CONFIG.get("clean_spikes"):
    for k in NAVN:
        NAVN[k], nfix = D.despike_daily(NAVN[k])
        ETF_FIXED[f"{k} NAV"] = nfix
for k, src in CONFIG.get("price_source", {}).items():
    if src == "nav" and k in NAVN and k in PXd:
        nav_k = NAVN[k].reindex(PXd.index.union(NAVN[k].index)).ffill().reindex(PXd.index)
        PXd[k] = nav_k.where(nav_k.notna(), PXd[k])
        print(f"{k}: returns from unit-normalised NAV ({nav_k.notna().mean():.0%} of trading days)")
PXm = PXd.resample("ME").last()
Rd = PXd.pct_change().dropna(how="all")
Rm = PXm.pct_change()
start_common = PXm.dropna().index[0]
Rm = Rm.loc[start_common:].dropna()
W = CONFIG["backtest"]["window"]
OOS = PXm.loc[start_common:].index[W:]
PX_OOS = PXm.loc[OOS]
RF_OOS = RF.reindex(OOS).ffill()
print(f"ETFs: {NAMES}")
print(f"Common data start: {start_common:%Y-%m}; out-of-sample: {OOS[0]:%Y-%m} to {OOS[-1]:%Y-%m} ({len(OOS)} months)")

## 8. Descriptive statistics

In [ ]:
from scipy import stats as sst
t1 = SEL.copy()
if "first_date" in t1:
    t1["first_date"] = pd.to_datetime(t1["first_date"]).dt.date
OUT.table(t1.set_index("sleeve"), "T01_etf_selection", "ETFs selected for each asset class (free Yahoo Finance data; liquidity screen on median daily traded value, Rs crore).")

desc = []
for k in KEYS:
    r = Rm[k]
    jb = sst.jarque_bera(r)
    desc.append({"ETF": NAMES[k], "mean_monthly_%": r.mean() * 100, "sd_monthly_%": r.std() * 100,
                 "ann_return_%": ((1 + r).prod() ** (12 / len(r)) - 1) * 100, "ann_vol_%": r.std() * np.sqrt(12) * 100,
                 "min_%": r.min() * 100, "max_%": r.max() * 100, "skewness": r.skew(), "excess_kurtosis": r.kurt(),
                 "jarque_bera": jb.statistic, "jb_p": jb.pvalue, "months": len(r)})
T2 = pd.DataFrame(desc).set_index("ETF")
OUT.table(T2, "T02_descriptive_stats", f"Descriptive statistics of monthly ETF returns, {Rm.index[0]:%b %Y}-{Rm.index[-1]:%b %Y}.")
T3 = Rm.rename(columns=NAMES).corr()
OUT.table(T3, "T03_correlations", "Correlation matrix of monthly ETF returns.")
fig = S.fig_corr(Rm, NAMES); OUT.figure(fig, "F03_correlation_heatmap", "Correlation of monthly ETF returns."); plt.show()

sm = SMETA.copy()
universe = pd.DataFrame({
    "stocks ever eligible": [len(SP.columns)],
    "eligible at OOS start": [int(SE.loc[OOS[0]].sum()) if OOS[0] in SE.index else np.nan],
    "delisted / stopped before end": [int(SP.loc[OOS].iloc[-1].isna().sum())],
    "share priced from Yahoo adj. close": [float((sm["price_source"] == "yahoo_adj").mean()) if "price_source" in sm else np.nan],
    "split/bonus corrections (bhavcopy fallback)": [int(sm.get("split_corrections", pd.Series([0])).sum())],
    "monthly returns > +100% (stock-months, raw)": [int((SP_RAW.pct_change(fill_method=None) > 1.0).sum().sum())],
    "monthly returns < -60% (stock-months, raw)": [int((SP_RAW.pct_change(fill_method=None) < -0.6).sum().sum())],
    "spike-reversals fixed (|r|>50% then reversal to within 20%)": [N_SPIKES],
    "monthly returns > +100% after cleaning": [int((SP.pct_change(fill_method=None) > 1.0).sum().sum())],
    "ETF daily bad ticks fixed (price + NAV)": [int(sum(ETF_FIXED.values()))]}).T
universe.columns = ["value"]
OUT.table(universe, "T04_stock_universe", "Direct-equity universe built from NSE month-end bhavcopies (survivorship-bias free).")
display(T2.round(3)); display(universe)

## 9. ETF tracking quality, premium/discount and liquidity

In [ ]:
te_rows = []
navw = NAVN                                 # unit-normalised NAV (same basis as split-adjusted price)
triw = S.wide_daily(etf_tri, "key", "tri") if len(etf_tri) else pd.DataFrame()
closew = S.wide_daily(etf_daily, "key", "close")   # split-adjusted, not dividend-adjusted (comparable with NAV)
volw = S.wide_daily(etf_daily, "key", "volume")
for k in KEYS:
    row = {"ETF": NAMES[k]}
    if k in triw and k in navw:
        row.update({f"nav_{a}": b for a, b in E.tracking_measures(navw[k].pct_change(), triw[k].pct_change()).items()})
    if k in triw:
        row.update(E.tracking_measures(closew[k].pct_change(), triw[k].pct_change()))
    if k in navw:
        row.update(E.premium_discount(closew[k], navw[k]))
    row.update(E.liquidity_measures(closew[k], volw[k]))
    te_rows.append(row)
T5 = pd.DataFrame(te_rows).set_index("ETF")
# show the days excluded as data errors so they can be checked against the source
bad_rows = []
for k in KEYS:
    if k not in triw:
        continue
    ri = triw[k].pct_change()
    for src, ser in (("market price", closew.get(k)), ("NAV", navw.get(k) if k in navw else None)):
        if ser is None:
            continue
        re_ = ser.pct_change()
        dd = pd.concat([re_, ri], axis=1).dropna()
        flag = dd[(dd.iloc[:, 0] - dd.iloc[:, 1]).abs() > 0.20]
        for d_, r_ in flag.iterrows():
            bad_rows.append({"ETF": NAMES[k], "series": src, "date": d_.date(), "ETF return": r_.iloc[0], "index return": r_.iloc[1],
                             "level": ser.loc[d_], "previous level": ser.shift(1).loc[d_]})
if bad_rows:
    BAD = pd.DataFrame(bad_rows)
    BAD.to_csv(f"{DIRS['tables']}/etf_data_error_days.csv", index=False)
    print("Days excluded from tracking error as data errors (|ETF - index| > 20% in one day):")
    display(BAD.round(4))
OUT.table(T5, "T05_tracking_liquidity", "Tracking error (TE1 mean absolute difference, TE2 SD of differences, TE3 regression standard error; annualised, price vs TRI; nav_ prefix = NAV vs TRI), premium/discount to NAV and liquidity.")
fig = S.fig_tracking(T5); OUT.figure(fig, "F10_tracking_premium", "Tracking error and premium/discount of the selected ETFs."); plt.show()
display(T5.T.round(4))

## 10. Dynamic conditional correlations (DCC-GARCH)

In [ ]:
try:
    DCC = E.dcc_correlations(Rd.loc[start_common:].dropna())
    eq = KEYS[0]
    pairs = [c for c in DCC.columns if c.startswith(eq + " /")]
    DCC_l = DCC[pairs].rename(columns=lambda c: " / ".join(NAMES[x] for x in c.split(" / ")))
    fig = S.fig_dcc(DCC_l, list(DCC_l.columns))
    OUT.figure(fig, "F04_dcc_correlations", f"DCC-GARCH(1,1) correlation of the {NAMES[eq]} ETF with other ETFs (a={DCC.attrs['dcc_a']:.3f}, b={DCC.attrs['dcc_b']:.3f})."); plt.show()
    DCC_l.resample("ME").mean().to_csv(f"{DIRS['tables']}/dcc_correlations_monthly.csv")
except Exception as e:
    print("DCC skipped:", e)

## 11. Strategy weights (out-of-sample, rolling window)

In [ ]:
BT = CONFIG["backtest"]
REBAL = E.rebalance_schedule(OOS, BT["rebalance"])
def schedules(window=BT["window"], cov=BT["main_cov"], rebal=REBAL, oos=OOS):
    out = {}
    for s in BT["strategies"]:
        out[s] = S.weight_schedule(s, cov, Rm, Rd, RF, oos, rebal, window, BT["weight_cap"], single_asset=KEYS[0],
                                   equity_assets=[k for k in KEYS if TYPES[k] == "eq_etf"])
    return out
SCHED = schedules()
dr = []
for s, w in SCHED.items():
    cov_full = E.cov_ledoit_wolf(Rm)
    dr.append({"strategy": LBL[s], "avg_diversification_ratio": np.mean([E.diversification_ratio(w.loc[d], cov_full) for d in w.index]),
               **{f"avg_w_{NAMES[k]}": w[k].mean() for k in KEYS}})
T6 = pd.DataFrame(dr).set_index("strategy")
OUT.table(T6, "T06_weights_diversification", "Average target weights and diversification ratio of ETF strategies.")
fig = S.fig_weights({LBL[s]: SCHED[s] for s in ("MinVar", "MaxSharpe", "RiskParity")}, NAMES)
OUT.figure(fig, "F09_weights", "Target weights of optimised ETF strategies at each rebalance date."); plt.show()
display(T6.round(3))

## 12. ETF portfolio simulations (investor size × broker × tax slab × friction level)

In [ ]:
def sel(df, **kw):
    # boolean-mask filter (avoids DataFrame.query, which breaks with numexpr + string dtypes on Colab)
    m = np.ones(len(df), dtype=bool)
    for k, v in kw.items():
        m &= (df[k] == v).to_numpy()
    return df[m]

def contrib_for(mode, amt, dates=OOS):
    return S.contributions(dates, mode, amt)

INVESTORS = [("lump", a) for a in CONFIG["investors"]["lump"]] + [("sip", a) for a in CONFIG["investors"]["sip"]]
def inv_label(m, a): return f"{'Lump' if m == 'lump' else 'SIP'} Rs {a:,.0f}" + ("/month" if m == "sip" else "")
MODES = ["frictionless", "costs", "costs_tax"]
etf_rows, ETF_RET = [], {}
t0 = time.time()
for (im, amt) in INVESTORS:
    c = contrib_for(im, amt)
    for broker in CONFIG["brokers"]:
        for mode in MODES:
            slabs = CONFIG["slabs"] if mode == "costs_tax" else [CONFIG["main_slab"]]
            if mode == "frictionless" and broker != CONFIG["main_broker"]:
                continue
            for slab in slabs:
                for s in BT["strategies"]:
                    res = S.run_one(PX_OOS, TYPES, SCHED[s], c, REBAL, mode, broker, slab, CONFIG)
                    m = E.sim_summary(res, RF_OOS)
                    etf_rows.append({"investor": inv_label(im, amt), "inv_mode": im, "amount": amt, "broker": broker,
                                     "mode": mode, "slab": slab, "strategy": s, **m})
                    ETF_RET[(im, amt, broker, mode, slab, s)] = res.returns
ETF_ALL = pd.DataFrame(etf_rows)
ETF_ALL.to_csv(f"{DIRS['tables']}/etf_all_scenarios.csv", index=False)
print(f"{len(ETF_ALL)} ETF simulations in {time.time() - t0:.0f}s")

MI, ML = CONFIG["main_investor"], CONFIG["main_lump"]
def main_key(s, mode="costs_tax", inv=MI, broker=CONFIG["main_broker"], slab=CONFIG["main_slab"]):
    return (inv["mode"], inv["amount"], broker, mode, slab if mode == "costs_tax" else CONFIG["main_slab"], s)
cols = ["ann_return", "ann_vol", "sharpe", "sortino", "max_drawdown", "cvar95_monthly", "ceq", "annual_turnover",
        "cost_pct_contrib", "tax_pct_contrib", "tax_pct_gain", "wealth_multiple_post_tax", "xirr_post_tax"]
sub = ETF_ALL[(ETF_ALL["broker"] == CONFIG["main_broker"]) & (ETF_ALL["slab"] == CONFIG["main_slab"])
              & (((ETF_ALL["inv_mode"] == MI["mode"]) & (ETF_ALL["amount"] == MI["amount"]))
                 | ((ETF_ALL["inv_mode"] == ML["mode"]) & (ETF_ALL["amount"] == ML["amount"])))]
T7 = sub.assign(strategy=sub["strategy"].map(LBL)).set_index(["investor", "mode", "strategy"])[cols]
OUT.table(T7, "T07_etf_performance_main", f"Out-of-sample performance of ETF strategies for the main investors ({inv_label(MI['mode'], MI['amount'])}; {inv_label(ML['mode'], ML['amount'])}), {CONFIG['main_broker']} brokerage, {CONFIG['main_slab']:.0%} slab: frictionless, with costs, and with costs and tax.")
display(T7.round(4))

## 13. Random direct-equity portfolios (survivorship-bias-free)

In [ ]:
SC = CONFIG["stocks"]
SP_OOS = SP.reindex(OOS)
GF = None
stock_rows, STOCK_RET = [], {}
t0 = time.time()
SETS = {N: S.draw_stock_sets(SP, SE, OOS[0], N, SC["draws"], SC["seed"]) for N in SC["N_list"]}
# (a) main investor, full number of draws, frictionless and costs+tax
for N, sets in SETS.items():
    for j, ids in enumerate(sets):
        for mode in ("frictionless", "costs_tax"):
            res = S.run_stock_portfolio(SP_OOS, ids, contrib_for(MI["mode"], MI["amount"]), REBAL, mode, CONFIG["main_broker"], CONFIG)
            m = E.sim_summary(res, RF_OOS)
            stock_rows.append({"N": N, "draw": j, "investor": inv_label(MI["mode"], MI["amount"]), "inv_mode": MI["mode"],
                               "amount": MI["amount"], "broker": CONFIG["main_broker"], "mode": mode, **m})
            if mode == "costs_tax":
                STOCK_RET[(N, j)] = res.returns
    print(f"  N={N}: {len(sets)} draws done ({time.time() - t0:.0f}s)")
# (b) investor-size x broker grid on a subset of draws (common random numbers)
for (im, amt) in INVESTORS:
    for broker in CONFIG["brokers"]:
        if (im, amt, broker) == (MI["mode"], MI["amount"], CONFIG["main_broker"]):
            continue
        for N, sets in SETS.items():
            for j, ids in enumerate(sets[:SC["draws_grid"]]):
                res = S.run_stock_portfolio(SP_OOS, ids, contrib_for(im, amt), REBAL, "costs_tax", broker, CONFIG)
                stock_rows.append({"N": N, "draw": j, "investor": inv_label(im, amt), "inv_mode": im, "amount": amt,
                                   "broker": broker, "mode": "costs_tax", **E.sim_summary(res, RF_OOS)})
    print(f"  grid {inv_label(im, amt)} done ({time.time() - t0:.0f}s)")
STK_ALL = pd.DataFrame(stock_rows)
STK_ALL.to_csv(f"{DIRS['tables']}/stock_all_draws.csv", index=False)
print(f"{len(STK_ALL)} stock-portfolio simulations in {time.time() - t0:.0f}s")

In [ ]:
main_stk = STK_ALL[(STK_ALL["investor"] == inv_label(MI["mode"], MI["amount"])) & (STK_ALL["broker"] == CONFIG["main_broker"])]
etf_main = {s: sel(ETF_ALL, inv_mode=MI["mode"], amount=MI["amount"], broker=CONFIG["main_broker"], mode="costs_tax", slab=CONFIG["main_slab"], strategy=s).iloc[0]
            for s in BT["strategies"]}
rows = []
for mode in ("frictionless", "costs_tax"):
    for N in SC["N_list"]:
        d = main_stk[(main_stk["N"] == N) & (main_stk["mode"] == mode)]
        e = sel(ETF_ALL, inv_mode=MI["mode"], amount=MI["amount"], broker=CONFIG["main_broker"], mode=mode, slab=CONFIG["main_slab"], strategy="EW").iloc[0]
        e1 = sel(ETF_ALL, inv_mode=MI["mode"], amount=MI["amount"], broker=CONFIG["main_broker"], mode=mode, slab=CONFIG["main_slab"], strategy="EQ100").iloc[0]
        e2 = sel(ETF_ALL, inv_mode=MI["mode"], amount=MI["amount"], broker=CONFIG["main_broker"], mode=mode, slab=CONFIG["main_slab"], strategy="EQ_EW").iloc[0]
        rows.append({"mode": mode, "N stocks": N, "draws": len(d),
                     "Sharpe p5": d["sharpe"].quantile(.05), "Sharpe median": d["sharpe"].median(), "Sharpe p95": d["sharpe"].quantile(.95),
                     "vol median": d["ann_vol"].median(), "MDD median": d["max_drawdown"].median(),
                     "cash drag (avg cash weight) median": d["avg_cash_weight"].median(),
                     "cost % contrib median": d["cost_pct_contrib"].median(), "tax % contrib median": d["tax_pct_contrib"].median(),
                     "1/N ETF Sharpe": e["sharpe"], "% draws below 1/N ETF Sharpe": (d["sharpe"] < e["sharpe"]).mean() * 100,
                     "% draws with higher vol than 1/N ETF": (d["ann_vol"] > e["ann_vol"]).mean() * 100,
                     "Nifty 50 ETF Sharpe": e1["sharpe"], "% draws below Nifty 50 ETF Sharpe": (d["sharpe"] < e1["sharpe"]).mean() * 100,
                     "Equity ETFs 1/N Sharpe": e2["sharpe"], "% draws below Equity ETFs 1/N Sharpe": (d["sharpe"] < e2["sharpe"]).mean() * 100,
                     "% draws with higher vol than Nifty 50 ETF": (d["ann_vol"] > e1["ann_vol"]).mean() * 100})
T8 = pd.DataFrame(rows).set_index(["mode", "N stocks"])
OUT.table(T8, "T08_stocks_vs_etf", f"Random N-stock portfolios versus ETF portfolios (equity-only: Nifty 50 ETF and Equity ETFs 1/N; multi-asset: 1/N ETF), {inv_label(MI['mode'], MI['amount'])}, {CONFIG['main_broker']} brokerage.")
display(T8.round(3))
fig = S.fig_sharpe_dist(main_stk[main_stk["mode"] == "costs_tax"][["N", "sharpe"]], {LBL[s]: etf_main[s]["sharpe"] for s in ("EQ100", "EQ_EW", "EW")})
OUT.figure(fig, "F05_sharpe_distribution", "Distribution of net Sharpe ratios of random N-stock portfolios against ETF strategies (SIP investor, costs and tax)."); plt.show()
N_band = 10 if 10 in SC["N_list"] else SC["N_list"][0]
band = pd.DataFrame({j: STOCK_RET[(N_band, j)] for j in range(len(SETS[N_band]))})
ret_main = {LBL[s]: ETF_RET[main_key(s)] for s in BT["strategies"]}
fig = S.fig_cumulative(ret_main, band, f"Random {N_band}-stock")
OUT.figure(fig, "F01_cumulative_growth", "Growth of Rs 1 (time-weighted, net of costs and tax): ETF strategies vs random 10-stock portfolios."); plt.show()

### 13b. Like-for-like check and risk decomposition
The main universe (500 most-traded stocks) includes mid and small caps. Here stocks are also drawn from the 100 most-traded stocks only. Ranking is by traded value (free data has no market cap), so this universe is liquid but still includes high-beta, heavily traded names.

Table T16 separates two effects: **systematic** (an equal-weighted basket of the whole universe vs the cap-weighted Nifty 50 ETF) and **diversifiable** (a small random N-stock portfolio vs that whole-universe basket). The universe baskets are frictionless and not investable by a small investor; they are reference points only.

In [ ]:
T15 = None
if SRANK is not None:
    LC = SC["large_cap_top_n"]
    ELIG_LC = (SRANK <= LC).reindex(columns=SP.columns).fillna(False)
    rows, LC_RET = [], {}
    for N in SC["N_list"]:
        sets_lc = S.draw_stock_sets(SP, ELIG_LC, OOS[0], N, SC["draws"], SC["seed"] + 7)
        for j, ids in enumerate(sets_lc):
            res = S.run_stock_portfolio(SP_OOS, ids, contrib_for(MI["mode"], MI["amount"]), REBAL, "costs_tax", CONFIG["main_broker"], CONFIG)
            rows.append({"N": N, "draw": j, **E.sim_summary(res, RF_OOS)})
            LC_RET[(N, j)] = res.returns
    LCD = pd.DataFrame(rows)
    LCD.to_csv(f"{DIRS['tables']}/stock_largecap_draws.csv", index=False)
    e1, e2 = etf_main["EQ100"], etf_main["EQ_EW"]
    out = []
    for N in SC["N_list"]:
        d = LCD[LCD["N"] == N]
        out.append({"N stocks": N, "draws": len(d), "Sharpe p5": d["sharpe"].quantile(.05), "Sharpe median": d["sharpe"].median(),
                    "Sharpe p95": d["sharpe"].quantile(.95), "vol median": d["ann_vol"].median(), "MDD median": d["max_drawdown"].median(),
                    "XIRR median": d["xirr_post_tax"].median(),
                    "% draws below Nifty 50 ETF Sharpe": (d["sharpe"] < e1["sharpe"]).mean() * 100,
                    "% draws below Equity ETFs 1/N Sharpe": (d["sharpe"] < e2["sharpe"]).mean() * 100,
                    "% draws with higher vol than Nifty 50 ETF": (d["ann_vol"] > e1["ann_vol"]).mean() * 100,
                    "% draws with deeper MDD than Nifty 50 ETF": (d["max_drawdown"] < e1["max_drawdown"]).mean() * 100})
    T15 = pd.DataFrame(out).set_index("N stocks")
    OUT.table(T15, "T15_largecap_universe", f"Random N-stock portfolios drawn from the {LC} most-traded stocks (large-cap, Nifty 100-like universe) versus equity ETFs, {inv_label(MI['mode'], MI['amount'])}, net of costs and tax. Nifty 50 ETF Sharpe {e1['sharpe']:.2f}, vol {e1['ann_vol']:.1%}, MDD {e1['max_drawdown']:.1%}.")
    display(T15.round(3))

    # ---- T16: universe equal-weight baskets and risk decomposition ----
    def beta_vs(r, m):
        a, b = r.align(m, join="inner")
        return float(np.cov(a, b)[0, 1] / np.var(b, ddof=1))
    mret = ETF_RET[main_key("EQ100")]
    dec = []
    def dec_row(name, r=None, m=None):
        if m is None:
            m = E.perf_metrics(r, RF_OOS)
        dec.append({"portfolio": name, "ann_return": m["ann_return"], "ann_vol": m["ann_vol"], "sharpe": m["sharpe"],
                    "max_drawdown": m["max_drawdown"], "beta vs Nifty 50 ETF": beta_vs(r, mret) if r is not None else np.nan})
    dec_row("Nifty 50 ETF (cap-weighted)", mret)
    UNIV_RET = {}
    for lbl, elig in ((f"top-{LC}", ELIG_LC), (f"top-{SC['top_n']}", SE)):
        ids_u = [c for c in SP.columns if bool(elig.loc[OOS[0], c]) and np.isfinite(SP.loc[OOS[0], c])]
        res_u = S.run_stock_portfolio(SP_OOS, ids_u, contrib_for(MI["mode"], MI["amount"]), REBAL, "frictionless", CONFIG["main_broker"], CONFIG)
        UNIV_RET[lbl] = res_u.returns
        dec_row(f"All {len(ids_u)} {lbl} stocks, equal-weight (frictionless reference)", res_u.returns)
    for lbl, src, store in ((f"top-{LC}", LCD, LC_RET), (f"top-{SC['top_n']}", main_stk[main_stk["mode"] == "costs_tax"], STOCK_RET)):
        for N in SC["N_list"]:
            keys = [k for k in store if k[0] == N]
            betas = [beta_vs(store[k], mret) for k in keys]
            d = src[src["N"] == N]
            dec.append({"portfolio": f"Random {N} stocks from {lbl} (median of draws)", "ann_return": d["ann_return"].median(),
                        "ann_vol": d["ann_vol"].median(), "sharpe": d["sharpe"].median(), "max_drawdown": d["max_drawdown"].median(),
                        "beta vs Nifty 50 ETF": float(np.median(betas))})
    T16 = pd.DataFrame(dec).set_index("portfolio")
    OUT.table(T16, "T16_risk_decomposition", f"Risk decomposition, {inv_label(MI['mode'], MI['amount'])}: cap-weighted Nifty 50 ETF vs equal-weighted whole-universe baskets (systematic difference) vs random N-stock portfolios (adds diversifiable risk). Random portfolios net of costs and tax; universe baskets frictionless.")
    display(T16.round(3))
else:
    print("stock_rank.csv.gz not found - re-run Section 6d (uses cached downloads) to enable this check.")

## 14. Statistical tests
Ledoit–Wolf (2008) Sharpe test, Ledoit–Wolf (2011) variance test, Romano–Wolf (2005) stepdown.

In [ ]:
TS = CONFIG["tests"]
ex = {s: (ETF_RET[main_key(s)] - RF_OOS.reindex(ETF_RET[main_key(s)].index)).dropna() for s in BT["strategies"]}
rows = []
for bench in ("EQ100", "EW"):
    for s in BT["strategies"]:
        if s == bench: continue
        a, b = ex[s].align(ex[bench], join="inner")
        sh = E.ledoit_wolf_sharpe_test(a.values, b.values, TS["block"], TS["bootstrap_M"], TS["seed"])
        va = E.ledoit_wolf_variance_test(a.values, b.values, TS["block"], TS["bootstrap_M"], TS["seed"])
        rows.append({"benchmark": LBL[bench], "strategy": LBL[s], "Sharpe diff (ann.)": sh["diff_annualised"],
                     "Sharpe p (LW2008)": sh["p_value"], "log variance ratio": va["diff"], "variance p (LW2011)": va["p_value"]})
T9 = pd.DataFrame(rows).set_index(["benchmark", "strategy"])
rw = E.romano_wolf_sharpe(ex["EW"].values, {LBL[s]: ex[s].values for s in BT["strategies"] if s != "EW"},
                          TS["block"], TS["bootstrap_M"], TS["seed"]).set_index("strategy")
T9 = T9.join(rw["rw_adj_p"].rename("Romano-Wolf adj. p (vs 1/N)"), on="strategy")
T9.loc[T9.index.get_level_values("benchmark") != LBL["EW"], "Romano-Wolf adj. p (vs 1/N)"] = np.nan
OUT.table(T9, "T09_tests_etf", f"Tests of ETF strategies against the Nifty 50 ETF and the 1/N ETF portfolio (net of costs and tax; circular block bootstrap, b={TS['block']}, M={TS['bootstrap_M']}).")
display(T9.round(4))

# ETF portfolios against every random stock draw (equity-only and multi-asset benchmarks)
rows = []
for bench in ("EQ100", "EQ_EW", "EW"):
    for N in SC["N_list"]:
        pv, diffs = [], []
        for j in range(len(SETS[N])):
            r = STOCK_RET[(N, j)]
            se = (r - RF_OOS.reindex(r.index)).dropna()
            a, b = ex[bench].align(se, join="inner")
            if len(a) < 24 or b.std() == 0: continue
            t = E.ledoit_wolf_sharpe_test(a.values, b.values, TS["block"], TS["stock_test_M"], TS["seed"])
            pv.append(t["p_value"]); diffs.append(t["diff_annualised"])
        pv, diffs = np.array(pv), np.array(diffs)
        rows.append({"ETF benchmark": LBL[bench], "N stocks": N, "draws tested": len(pv),
                     "median Sharpe diff (ETF - stocks)": np.median(diffs),
                     "% ETF significantly better (p<0.05)": np.mean((pv < TS["alpha"]) & (diffs > 0)) * 100,
                     "% stocks significantly better (p<0.05)": np.mean((pv < TS["alpha"]) & (diffs < 0)) * 100,
                     "% not significant": np.mean(pv >= TS["alpha"]) * 100})
T10 = pd.DataFrame(rows).set_index(["ETF benchmark", "N stocks"])
OUT.table(T10, "T10_tests_etf_vs_stocks", "Ledoit-Wolf Sharpe tests of ETF portfolios against each random N-stock portfolio (net of costs and tax).")
display(T10.round(2))

## 15. Cost and tax erosion by investor size, broker and tax slab

In [ ]:
rows = []
for (im, amt) in INVESTORS:
    for broker in CONFIG["brokers"]:
        e = sel(ETF_ALL, inv_mode=im, amount=amt, broker=broker, mode="costs_tax", slab=CONFIG["main_slab"], strategy="EW")
        st = sel(STK_ALL, inv_mode=im, amount=amt, broker=broker, mode="costs_tax", N=N_band)
        if e.empty or st.empty: continue
        e = e.iloc[0]
        rows.append({"investor": inv_label(im, amt), "broker": broker,
                     "1/N ETF Sharpe": e["sharpe"], f"{N_band}-stock median Sharpe": st["sharpe"].median(),
                     "1/N ETF cost %": e["cost_pct_contrib"] * 100, f"{N_band}-stock cost % (median)": st["cost_pct_contrib"].median() * 100,
                     "1/N ETF tax %": e["tax_pct_contrib"] * 100, f"{N_band}-stock tax % (median)": st["tax_pct_contrib"].median() * 100,
                     "1/N ETF tax % of gains": e["tax_pct_gain"] * 100, f"{N_band}-stock tax % of gains (median)": st["tax_pct_gain"].median() * 100,
                     "1/N ETF XIRR post-tax": e["xirr_post_tax"], f"{N_band}-stock XIRR median": st["xirr_post_tax"].median(),
                     "1/N ETF cash weight": e["avg_cash_weight"], f"{N_band}-stock cash weight (median)": st["avg_cash_weight"].median()})
T11 = pd.DataFrame(rows).set_index(["investor", "broker"])
OUT.table(T11, "T11_erosion_by_investor", f"Costs and taxes as % of contributions, cash drag and net performance by investor size and broker: 1/N ETF vs median random {N_band}-stock portfolio ({CONFIG['main_slab']:.0%} slab).")
display(T11.round(3))
er = []
for (inv, broker), r in T11.iterrows():
    if broker != "flat": continue
    er.append({"investor": inv, "portfolio": "1/N ETF", "cost_pct": r["1/N ETF cost %"], "tax_pct": r["1/N ETF tax %"]})
    er.append({"investor": inv, "portfolio": f"{N_band} stocks", "cost_pct": r[f"{N_band}-stock cost % (median)"], "tax_pct": r[f"{N_band}-stock tax % (median)"]})
if er:
    fig = S.fig_erosion(pd.DataFrame(er))
    OUT.figure(fig, "F06_cost_tax_erosion", f"Costs (solid) and taxes (hatched) as % of contributions with Rs 20 flat brokerage: 1/N ETF vs median {N_band}-stock portfolio."); plt.show()

T12 = (ETF_ALL[(ETF_ALL["mode"] == "costs_tax") & (ETF_ALL["broker"] == CONFIG["main_broker"])
              & (ETF_ALL["inv_mode"] == MI["mode"]) & (ETF_ALL["amount"] == MI["amount"])]
       .assign(strategy=lambda d: d["strategy"].map(LBL))
       .pivot_table(index="strategy", columns="slab", values=["sharpe", "xirr_post_tax", "tax_pct_contrib"]))
T12.columns = [f"{a} | slab {b:.0%}" for a, b in T12.columns]
OUT.table(T12, "T12_slab_scenarios", "ETF strategies under 5%, 20% and 30% income-tax slabs (slab affects debt ETF gains and short-term gold/international gains).")
display(T12.round(4))

## 16. Sub-period analysis

In [ ]:
sp = {k: v for k, v in CONFIG["subperiods"].items()}
ret_sp = dict(ret_main)
ret_sp[f"Random {N_band}-stock (cross-sectional median monthly return)"] = band.median(axis=1)
T13 = S.subperiod_table(ret_sp, RF_OOS, sp)
OUT.table(T13.set_index(["subperiod", "strategy"]), "T13_subperiods", "Performance by sub-period (main SIP investor, net of costs and tax).")
display(T13.round(3))

## 17. Robustness checks

In [ ]:
rows = []
c_main = contrib_for(MI["mode"], MI["amount"])
def add(check, variant, s, res):
    m = E.sim_summary(res, RF_OOS.reindex(res.returns.index))
    rows.append({"check": check, "variant": variant, "strategy": LBL[s], "sharpe": m["sharpe"], "ann_return": m["ann_return"],
                 "max_drawdown": m["max_drawdown"], "xirr_post_tax": m["xirr_post_tax"], "cost_pct_contrib": m["cost_pct_contrib"]})
base_args = dict(mode="costs_tax", broker=CONFIG["main_broker"], slab=CONFIG["main_slab"], cfg=CONFIG)
ROB = CONFIG["robustness"]
# (1) covariance estimator
for cov in BT["cov_methods"]:
    sch = SCHED if cov == BT["main_cov"] else schedules(cov=cov)
    for s in ("MinVar", "MaxSharpe", "RiskParity"):
        add("covariance", cov, s, S.run_one(PX_OOS, TYPES, sch[s], c_main, REBAL, **base_args))
# (2) estimation window (common OOS start = longest window, for comparability)
wmax = max(ROB["windows"])
oos_w = PXm.loc[start_common:].index[wmax:]
for w in ROB["windows"]:
    rb = E.rebalance_schedule(oos_w, BT["rebalance"])
    sch = schedules(window=w, rebal=rb, oos=oos_w)
    for s in ("EW", "MinVar", "MaxSharpe"):
        add("window (common start)", f"{w} months", s, S.run_one(PXm.loc[oos_w], TYPES, sch[s], contrib_for(MI["mode"], MI["amount"], oos_w), rb, **base_args))
# (3) rebalancing frequency
for fq in ROB["rebalance"]:
    rb = set() if fq == "T5" else E.rebalance_schedule(OOS, fq)
    sch = schedules(rebal=E.rebalance_schedule(OOS, "A") if fq == "T5" else rb)
    for s in ("EW", "MinVar"):
        add("rebalancing", "5% threshold" if fq == "T5" else fq, s,
            S.run_one(PX_OOS, TYPES, sch[s], c_main, rb, threshold=0.05 if fq == "T5" else None, **base_args))
# (4) extra transaction costs
for bps in ROB["extra_bps"]:
    for s in ("EQ100", "EW", "MinVar"):
        add("extra cost", f"+{bps} bps", s, S.run_one(PX_OOS, TYPES, SCHED[s], c_main, REBAL, extra_bps=bps, **base_args))
# (5) historical tax rules with grandfathering
gf = PXm.loc[:"2018-01-31"].iloc[-1].to_dict() if PXm.index[0] <= pd.Timestamp("2018-01-31") else None
for s in ("EQ100", "EW", "MinVar"):
    add("tax regime", "historical (date-dependent)", s, S.run_one(PX_OOS, TYPES, SCHED[s], c_main, REBAL, regime="historical", gf_prices=gf, **base_args))
    add("tax regime", "current FY2026-27", s, S.run_one(PX_OOS, TYPES, SCHED[s], c_main, REBAL, **base_args))
# (6) flat brokerage
for s in ("EQ100", "EW", "MinVar"):
    add("brokerage", "Rs 20 flat per order", s, S.run_one(PX_OOS, TYPES, SCHED[s], c_main, REBAL, mode="costs_tax", broker="flat", slab=CONFIG["main_slab"], cfg=CONFIG))
# (6b) SIP order rule: one order per month (most underweight ETF) vs split across all ETFs, zero brokerage
import copy
cfg_single = copy.deepcopy(CONFIG); cfg_single["sip_single_order_brokers"] = ["zero", "flat"]
for s in ("EW", "MinVar", "RiskParity"):
    add("SIP order rule", "split across all holdings (main, zero brokerage)", s, S.run_one(PX_OOS, TYPES, SCHED[s], c_main, REBAL, **base_args))
    add("SIP order rule", "one order per month (zero brokerage)", s,
        S.run_one(PX_OOS, TYPES, SCHED[s], c_main, REBAL, mode="costs_tax", broker=CONFIG["main_broker"], slab=CONFIG["main_slab"], cfg=cfg_single))
# (6c) cleaned vs raw stock prices: median Sharpe of the same random 10-stock draws
if CONFIG.get("clean_spikes") and N_SPIKES:
    SPR_OOS = SP_RAW.reindex(OOS)
    for lbl_, panel in (("cleaned (main)", SP_OOS), ("raw, uncleaned", SPR_OOS)):
        sh = [E.sim_summary(S.run_stock_portfolio(panel, ids, c_main, REBAL, "costs_tax", CONFIG["main_broker"], CONFIG), RF_OOS)
              for ids in SETS[N_band][:SC["draws_grid"]]]
        rows.append({"check": "stock price cleaning", "variant": lbl_, "strategy": f"Random {N_band} stocks (median)",
                     "sharpe": np.median([m["sharpe"] for m in sh]), "ann_return": np.median([m["ann_return"] for m in sh]),
                     "max_drawdown": np.median([m["max_drawdown"] for m in sh]), "xirr_post_tax": np.median([m["xirr_post_tax"] for m in sh]),
                     "cost_pct_contrib": np.median([m["cost_pct_contrib"] for m in sh])})
# (7) debt sleeve priced at market price instead of NAV
if CONFIG.get("price_source"):
    PXm_mkt = PXd_MKT.resample("ME").last()
    for s in ("EW", "MinVar", "RiskParity"):
        add("debt pricing", "market price (thin trading)", s, S.run_one(PXm_mkt.loc[OOS], TYPES, SCHED[s], c_main, REBAL, **base_args))
        add("debt pricing", "NAV (main)", s, S.run_one(PX_OOS, TYPES, SCHED[s], c_main, REBAL, **base_args))
T14 = pd.DataFrame(rows).set_index(["check", "variant", "strategy"])
OUT.table(T14, "T14_robustness", "Robustness of ETF results (main SIP investor, net of costs and tax).")
display(T14.round(4))

## 18. Additional figures

In [ ]:
fig = S.fig_drawdowns(ret_main); OUT.figure(fig, "F02_drawdowns", "Drawdowns of ETF strategies (net of costs and tax)."); plt.show()
fig = S.fig_rolling_sharpe(ret_main, RF_OOS); OUT.figure(fig, "F08_rolling_sharpe", "Rolling 36-month Sharpe ratio of ETF strategies."); plt.show()
cloud = main_stk[(main_stk["mode"] == "frictionless") & (main_stk["N"] == N_band)]
pts = {LBL[s]: SCHED[s].mean() for s in ("EW", "MinVar", "MaxSharpe")}
fig = S.fig_frontier(Rm, NAMES, pts, cloud)
OUT.figure(fig, "F07_efficient_frontier", "Long-only ETF efficient frontier (full sample, Ledoit-Wolf covariance) with average strategy weights and realised random 10-stock portfolios."); plt.show()

## 19. Export, key results and run log

In [ ]:
wb = OUT.write_workbook()
e = {s: etf_main[s] for s in BT["strategies"]}
t8 = T8.loc["costs_tax"]
lines = [f"# Key results - run {RUN_ID}", "",
         f"Data mode: **{DATA_MODE}**" + (" (SMOKE TEST ONLY - NOT FOR THE PAPER)" if DATA_MODE == "synthetic" else ""),
         f"Out-of-sample period: {OOS[0]:%b %Y} to {OOS[-1]:%b %Y} ({len(OOS)} months). ETFs: {', '.join(NAMES.values())}.",
         f"Main investor: {inv_label(MI['mode'], MI['amount'])}, {CONFIG['main_broker']} brokerage, {CONFIG['main_slab']:.0%} slab, current tax rules.", "",
         "## ETF strategies (net of costs and tax)"]
for s in BT["strategies"]:
    lines.append(f"- {LBL[s]}: Sharpe {e[s]['sharpe']:.2f}, annual return {e[s]['ann_return']:.1%}, volatility {e[s]['ann_vol']:.1%}, max drawdown {e[s]['max_drawdown']:.1%}, post-tax XIRR {e[s]['xirr_post_tax']:.1%}.")
lines += ["", "## Random direct-equity portfolios vs 1/N ETF"]
for N, r in t8.iterrows():
    lines.append(f"- {N} stocks: median Sharpe {r['Sharpe median']:.2f} (5th-95th pct {r['Sharpe p5']:.2f} to {r['Sharpe p95']:.2f}); "
                 f"{r['% draws below 1/N ETF Sharpe']:.0f}% of draws had a lower Sharpe than the 1/N ETF; "
                 f"1/N ETF significantly better in {T10.loc[(LBL['EW'], N), '% ETF significantly better (p<0.05)']:.0f}% of LW tests; "
                 f"{r['% draws below Nifty 50 ETF Sharpe']:.0f}% of draws below the Nifty 50 ETF and {r['% draws below Equity ETFs 1/N Sharpe']:.0f}% below the equity-ETF 1/N.")
lines += ["", "## Equity-only comparison (like-for-like)"]
for N in SC["N_list"]:
    r10 = T10.loc[(LBL["EQ100"], N)]
    lines.append(f"- {N} stocks (top-500 universe) vs Nifty 50 ETF: median Sharpe difference (ETF - stocks) {r10['median Sharpe diff (ETF - stocks)']:+.2f}; "
                 f"ETF significantly better in {r10['% ETF significantly better (p<0.05)']:.0f}%, stocks in {r10['% stocks significantly better (p<0.05)']:.0f}% of tests.")
if T15 is not None:
    for N, r in T15.iterrows():
        lines.append(f"- {N} stocks (large-cap universe): median Sharpe {r['Sharpe median']:.2f}, {r['% draws below Nifty 50 ETF Sharpe']:.0f}% of draws below the Nifty 50 ETF, "
                     f"{r['% draws with higher vol than Nifty 50 ETF']:.0f}% with higher volatility, {r['% draws with deeper MDD than Nifty 50 ETF']:.0f}% with a deeper drawdown.")
if "T16" in globals():
    lines += ["", "## Risk decomposition (T16)"]
    for nm, r in T16.iterrows():
        lines.append(f"- {nm}: Sharpe {r['sharpe']:.2f}, volatility {r['ann_vol']:.1%}, max drawdown {r['max_drawdown']:.1%}, beta {r['beta vs Nifty 50 ETF']:.2f}.")
lines += ["", "## Optimised vs 1/N (Romano-Wolf adjusted p-values)"]
for st, r in T9.xs(LBL["EW"], level="benchmark").iterrows():
    lines.append(f"- {st}: Sharpe difference {r['Sharpe diff (ann.)']:+.2f}, LW p = {r['Sharpe p (LW2008)']:.3f}, RW adj. p = {r['Romano-Wolf adj. p (vs 1/N)']:.3f}.")
lines += ["", "## Files", f"- Tables: {DIRS['tables']} (CSV, LaTeX, {os.path.basename(wb)})", f"- Figures: {DIRS['figures']} (PNG 300 dpi, PDF)"]
txt = "\n".join(lines)
open(f"{RUN_DIR}/key_results.md", "w").write(txt)
open(f"{DIRS['manuscript']}/key_results_{RUN_ID}.md", "w").write(txt)

import importlib.metadata as md_
vers = {p: md_.version(p) for p in ["numpy", "pandas", "scipy", "scikit-learn", "matplotlib", "arch", "yfinance", "requests", "openpyxl"]
        if importlib.util.find_spec(p.replace("scikit-learn", "sklearn")) is not None}
log = {"run_id": RUN_ID, "data_mode": DATA_MODE, "quick_run": QUICK_RUN, "python": platform.python_version(),
       "platform": platform.platform(), "packages": vers, "runtime_minutes": round((time.time() - T0) / 60, 1),
       "oos_start": str(OOS[0].date()), "oos_end": str(OOS[-1].date()), "n_months": len(OOS),
       "etfs": SEL.to_dict(orient="records"), "n_etf_sims": len(ETF_ALL), "n_stock_sims": len(STK_ALL)}
json.dump(log, open(f"{DIRS['logs']}/run_log.json", "w"), indent=2, default=str)
if os.path.exists(MAN.path):
    MAN.table().to_csv(f"{DIRS['logs']}/download_manifest_snapshot.csv", index=False)
import shutil
zip_path = shutil.make_archive(RUN_DIR, "zip", RUN_DIR)
print(txt)
print("\nAll outputs:", RUN_DIR, "\nZip:", zip_path)
display(pd.DataFrame(OUT.index))